# Core Solveit UI & Cell Chrome (`CRAFT.js`)

> Self-contained UI modules: hotkey bootstrap, auto-hiding top nav, compact Monaco line height, launcher (`⌘⇧P`), idle composer, cell chains, action menus, cell omit, cell parts, insert preview, `craft_md.js` loader, and image drop.

## Core UI Templates (`#| export_ft`)

Static UI shells and SVG icons compiled at build time via `fastcore.xml` `FT` trees into `window.__craftTpl`.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
CELL_PATHS = dict(
    Code='M8 6l-6 6 6 6m8-12 6 6-6 6M14 4l-4 16',
    Note='M5 3h10l4 4v14H5z M15 3v5h4M8 12h8m-8 4h8',
    Raw='M3 5h18v14H3z M7 9v6',
    Prompt='M3 4h18v12H9l-6 4z M7 10h.01m5 0h.01m5 0h.01',
)
DESK_ORDER = ('Code', 'Note', 'Prompt', 'Raw')

TEMPLATES = merge(
    dict(
        mobile_phone_icon=svg_icon([ft('rect', x='7', y='2', width='10', height='20', rx='2'), 'M11 18h2'], size=15, sw=1.8),
        mobile_desk_icon=svg_icon([ft('rect', x='2', y='3', width='20', height='14', rx='2'), 'M8 21h8M12 17v4'], size=15, sw=1.8),
        mobile_toggle_btn=Button(id='craft-mobile-toggle', type='button'),
        launcher_root=Div(
            Div(
                Input(type='text', placeholder='Run a command or open a dialog/file…',
                      autocomplete='off', spellcheck='false', aria_label='Search commands and files'),
                Ul(role='listbox'),
                klass='solveit-launcher-card'),
            id='solveit-launcher', role='dialog', aria_modal='true', aria_label='Command palette'),
        action_menu=Div(id='craft-cell-action-menu', klass='craft-action-menu', popover='auto', role='menu'),
        insert_trigger=Button(id='craft-insert-preview', type='button', hidden=True, popovertarget='craft-insert-preview-menu'),
        insert_menu=Div(
            Small('New cell type'),
            *[Button(svg_icon(p, 15, 1.6), name, type='button', data_kind=name) for name, p in CELL_PATHS.items()],
            id='craft-insert-preview-menu', popover='auto'),
        desk_edge_below=Button(
            Span(svg_icon('M12 6v12M6 12h12', 12, 1.8), Span(klass='craft-desk-edge-kind'), klass='craft-desk-edge-pill'),
            type='button', klass='craft-desk-edge', data_edge='below',
            aria_label='Insert cell below (drag to pick type)'),
        desk_picker=Div(
            Div(klass='craft-desk-picker-head'),
            Div(*[Button(svg_icon(CELL_PATHS[t], 14, 1.8), Span(t), ft('kbd', t[0]),
                         type='button', role='menuitem', data_type=t) for t in DESK_ORDER],
                klass='craft-desk-picker-opts'),
            id='craft-desk-picker', role='menu'),
        action_trigger=Button('…', type='button', klass='craft-action-trigger', aria_haspopup='menu', aria_expanded='false', aria_controls='craft-cell-action-menu'),
        omit_marker=Button(type='button', klass='craft-omit-marker'),
        insert_placeholder=Div(klass='craft-insert-placeholder'),
        table_wrap=Div(klass='craft-table-wrap', tabindex='0', role='region', aria_label='Scrollable table'),
        table_copy_btn=Button('Copy TSV', type='button', klass='craft-table-copy', aria_label='Copy table as TSV'),
    ),
    {f'cell_icon_{k}': svg_icon(p, 15, 1.6) for k, p in CELL_PATHS.items()},
    {f'desk_icon_{k}': svg_icon(p, 11, 1.8) for k, p in CELL_PATHS.items()},
)

CONFIG = dict(
    cell_types=list(DESK_ORDER),
    key_to_type={t[0].lower(): t for t in DESK_ORDER},
    slash_map={k: t for t in DESK_ORDER for k in (t[0].lower(), t.lower())},
)

## Hotkey Bootstrap

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Hotkey manager bootstrap ──────────────────────────────────────
   The shared shortcut registry lives in hotkey_manager.js (served from
   ~/.craft-assets at /static/.craft-assets/hotkey_manager.js). Load it first so the sections below
   can register their shortcuts; registrations made before it finishes
   loading are queued and replayed by the manager. */
(() => {
  window.__hotkeyQueue ||= [];
  window.__hotkeyRegister = spec => { if (window.solveitHotkeys) window.solveitHotkeys.register(spec); else window.__hotkeyQueue.push(spec); };
  for (const [src, tag] of [['/static/.craft-assets/hotkey_manager.js', '1'], ['/static/.craft-assets/hotkey_builtins.js', 'builtins']])
    if (!document.querySelector(`script[data-solveit-hotkeys="${tag}"]`)) {
      const s = document.createElement('script');
      s.src = src; s.dataset.solveitHotkeys = tag;
      document.head.append(s);
    }
})();

## Mobile Mode Detection

Phones and narrow windows (≤820px or touch-only devices) receive `html.craft-mobile`. All mobile CSS and JS branches key off this class so desktop layout stays untouched.

In [ ]:
#| eval: false
#| export_js CRAFT.js
/* ── Mobile mode detection & view toggle ───────────────────────────────
   Auto-enables `html.craft-mobile` only on touch/mobile devices (`(hover: none)
   and (pointer: coarse)`), so shrinking a desktop window stays in desktop view.
   Users can explicitly switch between Desktop and Mobile at any time via the
   navbar toggle button (#craft-mobile-toggle), the launcher, or
   `window.craftMobile.setMode('auto'|'mobile'|'desktop')`. */
(() => {
  window.craftMobile?.disable?.();
  const {mountTpl, qs, on, store, patch, onDomChange} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal};
  const PREF_KEY = 'craft-mobile-mode';
  const mq = matchMedia('(hover: none) and (pointer: coarse)');
  const root = document.documentElement, listeners = new Set();
  let meta = qs('meta[name="viewport"]'), addedMeta = false, prevMeta = meta?.content;
  const VIEWPORT = 'width=device-width, initial-scale=1, viewport-fit=cover, interactive-widget=resizes-content';
  if (!meta) { meta = document.createElement('meta'); meta.name = 'viewport'; document.head.append(meta); addedMeta = true; }
  if (!/width\s*=\s*device-width/.test(meta.content || '')) meta.content = VIEWPORT;
  else {
    if (!/viewport-fit/.test(meta.content)) meta.content += ', viewport-fit=cover';
    if (!/interactive-widget/.test(meta.content)) meta.content += ', interactive-widget=resizes-content';
  }
  const iOS = /iP(hone|ad|od)/.test(navigator.userAgent) || (navigator.platform === 'MacIntel' && navigator.maxTouchPoints > 1);
  if (iOS && !/maximum-scale/.test(meta.content)) meta.content += ', maximum-scale=1';

  const readPref = () => { const v = store.get(PREF_KEY); return (v === 'mobile' || v === 'desktop') ? v : 'auto'; };
  let pref = readPref();
  const computeActive = () => pref === 'mobile' ? true : pref === 'desktop' ? false : mq.matches;

  const TPL = window.__craftTpl || {};
  const PHONE_SVG = TPL.mobile_phone_icon || '', DESK_SVG = TPL.mobile_desk_icon || '';

  const toggleBtn = mountTpl('mobile_toggle_btn');

  const syncToggleBtn = () => {
    const isMob = root.classList.contains('craft-mobile');
    const key = `${isMob ? 'm' : 'd'}:${pref}`;
    if (toggleBtn.dataset.state === key) return;
    const title = isMob
      ? `Switch to desktop view (current: ${pref === 'auto' ? 'auto mobile' : 'mobile'})`
      : `Switch to mobile view (current: ${pref === 'auto' ? 'auto desktop' : 'desktop'})`;
    patch(toggleBtn, {dataset: {state: key}, innerHTML: isMob ? DESK_SVG : PHONE_SVG, title, 'aria-label': title, 'aria-pressed': String(isMob)});
  };

  const positionToggleBtn = () => {
    if (root.classList.contains('craft-mobile')) return;
    Object.assign(toggleBtn.style, { left: 'auto', right: '10px', top: '6px' });
  };

  const mountToggleBtn = () => {
    if (!toggleBtn.isConnected && document.body) document.body.append(toggleBtn);
    syncToggleBtn();
    positionToggleBtn();
  };

  const apply = () => {
    const on = computeActive();
    const changed = root.classList.contains('craft-mobile') !== on;
    if (changed) {
      root.classList.toggle('craft-mobile', on);
      listeners.forEach(fn => { try { fn(on); } catch (err) { console.error('craftMobile listener', err); } });
    }
    syncToggleBtn();
  };

  const setMode = m => {
    pref = (m === 'mobile' || m === 'desktop') ? m : 'auto';
    store.set(PREF_KEY, pref === 'auto' ? null : pref);
    apply();
    return root.classList.contains('craft-mobile');
  };
  const toggle = () => setMode(root.classList.contains('craft-mobile') ? 'desktop' : 'mobile');
  on(toggleBtn, 'click', e => { e.preventDefault(); e.stopPropagation(); toggle(); }, sig);
  on(mq, 'change', apply, sig);
  let offDom = null;
  on(window, 'resize', positionToggleBtn, {passive: true, signal: ac.signal});
  const start = () => {
    mountToggleBtn();
    offDom = onDomChange(mountToggleBtn, {delay: 0});
  };
  apply();
  if (document.body) start(); else on(document, 'DOMContentLoaded', start, {once: true, signal: ac.signal});

  window.__hotkeyRegister?.({
    id: 'craft-toggle-mobile',
    label: 'Toggle mobile / desktop view',
    group: 'CRAFT · Layout',
    combo: '',
    handler: () => { toggle(); return true; },
  });

  window.craftMobile = {
    active: () => root.classList.contains('craft-mobile'),
    mode: () => pref,
    setMode,
    toggle,
    onChange: fn => { listeners.add(fn); return () => listeners.delete(fn); },
    reposition: positionToggleBtn,
    disable() {
      ac.abort();
      offDom?.();
      toggleBtn.remove();
      root.classList.remove('craft-mobile');
      if (addedMeta) meta.remove(); else if (prevMeta != null) meta.content = prevMeta;
      listeners.clear();
      delete window.craftMobile;
    },
  };
})();

## Top Nav, Compact Monaco, Launcher & Idle Composer

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Auto-hiding top nav ──────────────────────────────────────────────
   The reveal trigger is decoupled from the bar. The band starts BELOW the
   browser's own top-edge hotspot so a hidden browser toolbar doesn't fight
   it, and the bar stays pinned while the pointer is over the nav or inside
   an open UIkit drop, fading only after a grace period. */
(() => {
  const {qs, on} = window.__craftCMCore;
  const TRIGGER_TOP = 28;      // px: clear of the browser's top-edge hotspot
  const TRIGGER_BOTTOM = 76;   // px: bottom of the reveal band
  const HIDE_DELAY = 700;      // ms grace period before fading out

  const SEL = '.uk-container > nav.monster-navbar';
  const PIN = '.uk-drop, .uk-dropdown, [uk-drop], [uk-dropdown]';
  const MODAL_OPEN = '.uk-modal.uk-open, .hk-overlay.open, #solveit-launcher.open';

  const nav = () => qs(SEL);
  let timer = null, last = null;

  function wantOpen() {
    if (qs(MODAL_OPEN)) return false;
    if (!last) return false;
    if (last.y >= TRIGGER_TOP && last.y <= TRIGGER_BOTTOM) return true;
    const t = last.target;
    if (t instanceof Element) {
      if (nav()?.contains(t)) return true;
      if (t.closest(PIN)) return true;
    }
    return !!qs('.uk-drop.uk-open, .uk-dropdown.uk-open');
  }

  function sync() {
    const n = nav();
    if (!n) return;
    if (wantOpen()) {
      clearTimeout(timer);
      timer = null;
      n.classList.add('nav-shown');
    } else if (!timer) {
      timer = setTimeout(() => {
        timer = null;
        if (!wantOpen()) nav()?.classList.remove('nav-shown');
      }, HIDE_DELAY);
    }
  }

  on(document, 'mousemove', e => {
    // Phones: taps emit compat mousemoves, which would flash the bar at random. The mobile toolbar's ☰ drives it instead.
    if (window.craftMobile?.active()) return;
    last = { y: e.clientY, target: e.target };
    sync();
  }, { passive: true });

  on(document, 'focusin', e => {
    last = { y: -1, target: e.target };
    sync();
  });
})();

/* ── Compact Monaco line height ──────────────────────────────────── */
/* Code size and family come from CRAFT.css. Monaco line-height matches the
   code size; remeasure after webfonts load to keep cursor columns aligned. */
(() => {
  let tries = 0;
  const go = () => {
    const mon = window.monaco?.editor;
    if (!mon) return ++tries < 40 && setTimeout(go, 250);
    // Recompute character widths after the webfonts replace the fallback.
    if (document.fonts) Promise.all([
      document.fonts.load('10px "Craft Iosevka"'),
      document.fonts.load('italic 10px "Craft Iosevka"'),
      document.fonts.load('bold 10px "Craft Iosevka"'),
      document.fonts.load('italic bold 10px "Craft Iosevka"')
    ]).then(() => { mon.remeasureFonts(); mon.getEditors().forEach(ed => ed.layout()); })
      .catch(err => console.warn('Iosevka could not load; using monospace fallback.', err));
    const compact = ed => {
      try {
        const cs = getComputedStyle(document.documentElement);
        const fs = parseFloat(cs.getPropertyValue('--craft-code')) || ed.getOption(mon.EditorOption.fontSize);
        const lh = Math.round(fs * (parseFloat(cs.getPropertyValue('--craft-code-lh')) || 1.2));
        ed.updateOptions({ fontSize: fs, lineHeight: lh, fontFamily: cs.getPropertyValue('--craft-code-family').trim() || 'monospace' });
      } catch (e) {}
    };
    mon.onDidCreateEditor?.(compact);
    mon.getEditors().forEach(compact);
    try { mon.getDiffEditors().forEach(d => { const cs = getComputedStyle(document.documentElement); const fs = parseFloat(cs.getPropertyValue('--craft-code')) || 10; const lh = Math.round(fs * (parseFloat(cs.getPropertyValue('--craft-code-lh')) || 1.2)); d.updateOptions({ fontSize: fs, lineHeight: lh, fontFamily: cs.getPropertyValue('--craft-code-family').trim() || 'monospace' }); }); } catch (e) {}
  };
  go();
})();

/* ── Launcher (⌘⇧P) ──────────────────────────────────────────────────
   One fuzzy box over commands (every hotkey-registry action, custom and
   Solveit built-in) and dialogs/files (whatever the file tree has loaded).
   Enter runs or opens the selection; Esc or a click outside closes. */
(() => {
  const {el, mountTpl, qsa, on} = window.__craftCMCore;
  let root, input, list, results = [], sel = 0, pool = [], prevFocus;

  function score(q, text) {
    if (!q) return 1;
    const t = text.toLowerCase();
    let i = 0, s = 0, run = 0, last = -2;
    for (const ch of q) {
      const j = t.indexOf(ch, i);
      if (j < 0) return 0;
      run = j === last + 1 ? run + 1 : 0;
      s += 1 + run * 2 + (j === 0 || /[\s/_.-]/.test(t[j - 1]) ? 3 : 0);
      last = j; i = j + 1;
    }
    return s - t.length * 0.01;
  }

  const where = n => { try { const u = new URL(n.href, location.href); return u.searchParams.get('name') || u.searchParams.get('fp') || decodeURIComponent(u.pathname); } catch { return ''; } };

  function build() {
    ({root, input, list} = mountTpl('launcher_root', {refs: {input: 'input', list: 'ul'}}));
    on(root, 'mousedown', e => { if (e.target === root) close(); });
    on(input, {
      input: () => { sel = 0; render(); },
      keydown: e => {
        if (e.key === 'ArrowDown' || e.key === 'ArrowUp') { e.preventDefault(); e.stopPropagation(); sel = (sel + (e.key === 'ArrowDown' ? 1 : -1) + results.length) % Math.max(results.length, 1); paint(); }
        else if (e.key === 'Enter') { e.preventDefault(); e.stopPropagation(); run(results[sel]); }
        else if (e.key === 'Escape') { e.preventDefault(); e.stopPropagation(); close(); }
      },
    });
    on(list, 'click', e => { const li = e.target.closest('li[data-i]'); if (li) run(results[+li.dataset.i]); });
    document.body.append(root);
  }

  function render() {
    const q = input.value.trim().toLowerCase();
    results = pool.map(it => ({it, s: score(q, it.label + ' ' + it.detail)})).filter(r => r.s > 0).sort((a, b) => b.s - a.s).slice(0, 60).map(r => r.it);
    sel = Math.min(sel, Math.max(results.length - 1, 0));
    list.replaceChildren(...(results.length ? results.map((it, i) =>
      el('li', {cls: 'kind-' + it.kind, role: 'option', dataset: {i}},
        el('span', {cls: 'label'}, it.label), el('span', {cls: 'detail'}, it.detail), el('span', {cls: 'hint'}, it.hint))
    ) : [el('li', {cls: 'empty'}, 'No matches')]));
    paint();
  }

  function paint() {
    qsa('li[data-i]', list).forEach(li => { const on = +li.dataset.i === sel; li.setAttribute('aria-selected', String(on)); if (on) li.scrollIntoView({block: 'nearest'}); });
  }

  async function open() {
    if (!root) build();
    prevFocus = document.activeElement;
    const hk = window.solveitHotkeys;
    const cmds = (hk?.list?.() || []).filter(a => a.id !== 'launcher').map(a => ({kind: 'command', label: a.label, detail: a.group || '', hint: a.combo ? hk.formatCombo(a.combo) : '', go: () => hk.run(a.id)}));
    pool = cmds; root.classList.add('open'); input.value = ''; sel = 0; render(); input.focus();
    try {
      const entries = await window.solveitFileTree?.entries?.() || [];
      pool = cmds.concat(entries.map(n => ({kind: n.kind, label: n.name, detail: where(n), hint: n.kind, go: () => window.solveitFileTree.open(n)})));
      if (root.classList.contains('open')) render();
    } catch {}
  }

  function close() { if (!root?.classList.contains('open')) return; root.classList.remove('open'); prevFocus?.focus?.(); }
  function run(item) { if (!item) return; close(); setTimeout(() => Promise.resolve().then(item.go).catch(err => { console.error('Launcher action failed', err); alert('Action failed: ' + (err.message || err)); }), 0); }

  window.__hotkeyRegister?.({id: 'launcher', label: 'Open launcher', group: 'Launcher', combo: 'Mod+Shift+P', handler: () => { if (root?.classList.contains('open')) close(); else open(); return true; }});
})();
(() => {
  window.craftIdleComposer?.disable();
  if (window.CRAFT_IDLE_COMPOSER === false) return;
  const {qs, on, onDomChange} = window.__craftCMCore;
  let editor, originalFocus, subscriptions = [], requested = false, pending = false;
  const panel = () => document.getElementById('full_editor');
  const temporary = () => editor?.getModel()?.uri.path.endsWith('/temp');
  const hasDraft = () => !!window.edValue?.();
  const reveal = () => {
    requested = true;
    panel()?.classList.remove('craft-composer-idle');
    editor?.layout();
  };
  const unbind = () => {
    subscriptions.forEach(s => s.dispose());
    subscriptions = [];
    if (editor && originalFocus) editor.focus = originalFocus;
  };
  const sync = () => {
    pending = false;
    if (window.editor !== editor) {
      unbind();
      editor = window.editor;
      if (editor) {
        subscriptions.push(editor.onDidChangeModel(schedule));
        subscriptions.push(editor.onDidChangeModelContent(schedule));
      }
    }
    const root = panel();
    if (!root) return;
    const editingCell = Boolean(editor?.getModel() && !temporary()) || Boolean(qs('.editable[data-editing="true"]'));
    const emptyDialog = !qs('#dialog-container > .editable');
    const active = requested || (!insertionHolds.size && (emptyDialog || editingCell || hasDraft()));
    const idle = !active;
    if (root.classList.contains('craft-composer-idle') === idle) return;
    root.classList.toggle('craft-composer-idle', idle);
    if (active && editor) editor.layout();
  };
  const schedule = () => { if (!pending) { pending = true; queueMicrotask(sync); } };
  const tabClick = e => {
    const tab = e.target.closest('#full_editor .tabh');
    if (!tab) return;
    if (tab.id === 'tabh-code' && window.craftC2?.shouldInterceptCodeTab?.()) return;
    reveal();
  };
  let shortcutOpened = false, dismissTimer;
  const insertionHolds = new Set(), rootEl = document.documentElement;
  const clearInsertionHolds = () => { insertionHolds.clear(); rootEl.removeAttribute('data-craft-inserting'); };
  const holdForInsertion = () => {
    const token = Symbol();
    insertionHolds.add(token); rootEl.setAttribute('data-craft-inserting', '');
    return () => {
      if (!insertionHolds.delete(token) || insertionHolds.size) return;
      if (!hasDraft() && temporary()) hide();
      rootEl.removeAttribute('data-craft-inserting');
    };
  };
  const hide = () => {
    clearTimeout(dismissTimer);
    shortcutOpened = false;
    requested = false;
    sync();
  };
  const show = () => {
    if (window.editor !== editor) sync();
    const root = panel();
    if (!root || !editor?.getModel()) return false;
    clearTimeout(dismissTimer);
    clearInsertionHolds();
    shortcutOpened ||= root.classList.contains('craft-composer-idle');
    reveal();
    editor.focus();
    return true;
  };
  const dismissShortcut = e => {
    if (!shortcutOpened || e.key !== 'Escape' || e.isComposing || !panel()?.contains(e.target)) return;
    clearTimeout(dismissTimer);
    dismissTimer = setTimeout(() => {
      shortcutOpened = false;
      if (!temporary() || hasDraft() || qs('.editable[data-editing="true"]')) return;
      requested = false;
      sync();
    }, 0);
  };
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const offDom = onDomChange(schedule, {attrs: ['data-editing'], delay: 0});
  const settleEvent = ['htmx', 'after', 'settle'].join(window.htmx?.config?.metaCharacter || ':');
  on(window, 'keydown', dismissShortcut, cap);
  on(document, 'click', tabClick, cap);
  on(document, settleEvent, schedule, sig);
  window.craftIdleComposer = {
    reveal,
    hide,
    show,
    holdForInsertion,
    disable() {
      offDom();
      ac.abort();
      clearTimeout(dismissTimer);
      clearInsertionHolds();
      if (window.craftIdleComposer?.show === show) delete window.craftIdleComposer;
      unbind();
      panel()?.classList.remove('craft-composer-idle');
      editor?.layout();
    }
  };
  sync();
  window.__hotkeyRegister?.({id: 'composer-show', label: 'Show cell editor', group: 'Editor', combo: 'Shift+I', handler: e => {
    if (e?.isComposing || e?.target?.closest?.('#solveit-file-tree-panel, #solveit-temp-editor, [role="dialog"], .uk-modal.uk-open, .hk-overlay.open, #solveit-launcher.open')) return false;
    return window.craftIdleComposer?.show() === true;
  }});
})();

## Cell Chains, Action Menus, Omit, Parts, Insert Preview & Image Drop

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftCellChains?.disable();
  let frame;
  const sync = () => {
    frame = null;
    const all = [...document.querySelectorAll('#dialog-container > .editable')];
    const cells = all.filter(e => !e.hidden && getComputedStyle(e).display !== 'none');
    const type = e => e?.querySelector(':scope > .card-in')?.dataset.mtype;
    const joins = (a, b) => ['prompt', 'code', 'note'].includes(type(a)) && type(a) === type(b);
    const prev = new Set(cells.filter((e, i) => i > 0 && joins(cells[i-1], e)));
    const next = new Set(cells.filter((e, i) => i < cells.length-1 && joins(e, cells[i+1])));
    for (const e of all) {
      if (e.classList.contains('craft-chain-prev') !== prev.has(e)) e.classList.toggle('craft-chain-prev', prev.has(e));
      if (e.classList.contains('craft-chain-next') !== next.has(e)) e.classList.toggle('craft-chain-next', next.has(e));
    }
  };
  const offDom = window.__craftCMCore.onDomChange(sync, {attrs: ['class', 'style', 'hidden', 'data-mtype'], delay: 'raf'});
  window.craftCellChains = {
    sync,
    disable() {
      offDom();
      document.querySelectorAll('.craft-chain-prev,.craft-chain-next').forEach(e => e.classList.remove('craft-chain-prev', 'craft-chain-next'));
    }
  };
  sync();
})();

(() => {
  window.craftActionMenus?.disable();
  if (window.CRAFT_ACTION_MENUS === false) return;
  let current = null, frame = null;
  const {mountTpl, qsa, on, onDomChange} = window.__craftCMCore;
  const menu = mountTpl('action_menu');
  document.body.append(menu);
  const close = (restore = false) => {
    const trigger = current;
    current = null;
    trigger?.setAttribute('aria-expanded', 'false');
    if (menu.matches(':popover-open')) menu.hidePopover();
    if (restore && trigger?.isConnected) trigger.focus({preventScroll: true});
  };
  const open = trigger => {
    if (current === trigger) return close(true);
    close();
    const card = trigger.closest('.msg-card');
    const group = card.querySelector('.uk-card-header .uk-btn-group');
    if (!group) return;
    menu.replaceChildren();
    for (const source of group.querySelectorAll('button,a')) {
      if (source.hidden) continue;
      const label = source.title || source.querySelector('[title]')?.getAttribute('title') || source.getAttribute('aria-label');
      if (!label) continue;
      menu.append(window.__craftCMCore.el('button', {
        type: 'button', role: 'menuitem', disabled: source.disabled,
        dataset: /^Delete/i.test(label) ? {danger: ''} : {},
        onClick: e => {
          e.stopPropagation();
          close();
          if (source.isConnected) source.click();
          queueMicrotask(() => { if (document.activeElement === document.body && trigger.isConnected) trigger.focus({preventScroll: true}); });
        },
      }, label));
    }
    if (card.matches('.card-in[data-mtype="code"]') && window.craftC2) {
      const cellEl = card.closest('.editable');
      const cid = cellEl?.id?.replace(/^_/, '');
      if (cid) {
        menu.append(
          window.__craftCMCore.el('hr', {cls: 'craft-action-sep'}),
          window.__craftCMCore.el('button', {
            type: 'button', role: 'menuitem',
            onClick: e => { e.stopPropagation(); close(); window.craftC2.toggleLineNumbers?.(); },
          }, `Line numbers: ${window.craftC2.lineNumbers?.() ? 'On' : 'Off'}`),
        );
      }
    }
    current = trigger;
    trigger.setAttribute('aria-expanded', 'true');
    menu.setAttribute('aria-label', trigger.getAttribute('aria-label'));
    menu.showPopover();
    const r = trigger.getBoundingClientRect();
    menu.style.left = Math.max(8, Math.min(r.right - menu.offsetWidth, innerWidth - menu.offsetWidth - 8)) + 'px';
    menu.style.top = Math.max(8, Math.min(r.bottom + 4, innerHeight - menu.offsetHeight - 8)) + 'px';
    menu.querySelector('button:not(:disabled)')?.focus({preventScroll: true});
  };
  const sync = () => {
    frame = null;
    if (current && !current.isConnected) close();
    for (const group of document.querySelectorAll('#dialog-container .msg-card > .uk-card-header .uk-btn-group')) {
      const nav = group.closest('nav');
      if (!nav) continue;
      let shell = group;
      while (shell.parentElement !== nav && shell.parentElement) shell = shell.parentElement;
      if (shell === group && shell.parentElement !== nav) continue;
      shell.classList.add('craft-action-source');
      for (const button of nav.querySelectorAll(':scope > button[data-uk-toggle]')) button.classList.add('craft-old-action-toggle');
      if (nav.querySelector(':scope > .craft-action-trigger')) continue;
      const trigger = mountTpl('action_trigger');
      const card = nav.closest('.msg-card');
      trigger.setAttribute('aria-label', card.classList.contains('card-out') ? 'Output actions' : 'Input actions');
      on(trigger, {
        click: e => { e.stopPropagation(); open(trigger); },
        keydown: e => { if (e.key === 'ArrowDown') { e.preventDefault(); e.stopPropagation(); open(trigger); } },
      });
      nav.append(trigger);
    }
  };
  const keys = e => {
    if (!current) return;
    if (e.key === 'Escape') { e.preventDefault(); e.stopImmediatePropagation(); close(true); return; }
    if (!menu.contains(e.target)) return;
    e.stopImmediatePropagation();
    const items = qsa('button:not(:disabled)', menu);
    const i = items.indexOf(document.activeElement);
    if (['ArrowDown', 'ArrowUp', 'Home', 'End'].includes(e.key)) {
      e.preventDefault();
      const j = e.key === 'Home' ? 0 : e.key === 'End' ? items.length-1 : (i + (e.key === 'ArrowDown' ? 1 : -1) + items.length) % items.length;
      items[j]?.focus();
    }
  };
  const focus = e => { if (current && !menu.contains(e.target) && e.target !== current) close(); };
  const toggled = () => { if (!menu.matches(':popover-open') && current) close(); };
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const offDom = onDomChange(sync, {delay: 'raf', inDialog: true});
  on(document, 'keydown', keys, cap);
  on(document, {focusin: focus, 'htmx:beforeSwap': () => close()}, sig);
  on(menu, 'toggle', toggled, sig);
  window.craftActionMenus = {
    sync,
    close,
    disable() {
      offDom();
      close();
      ac.abort();
      qsa('.craft-action-trigger').forEach(e => e.remove());
      qsa('.craft-action-source,.craft-old-action-toggle').forEach(e => e.classList.remove('craft-action-source', 'craft-old-action-toggle'));
      menu.remove();
    }
  };
  sync();
})();


(() => {
  const {mountTpl, qsa, on, onDomChange} = window.__craftCMCore;
  const preserved = qsa('.editable.craft-omitted').map(e => e.id);
  window.craftOmit?.disable();
  const omitted = new Set(preserved), markers = new Map();
  let frame, approach = 1;
  const root = () => document.getElementById('dialog-container');
  const cells = () => qsa(':scope > .editable', root());
  const eligible = e => !e.hidden && !e.matches('.filter-hidden,.heading-hidden,.hidden');
  const marker = () => document.activeElement?.closest('.craft-omit-marker');
  const blocked = () => window.__craftCMCore.typingBlocked();
  const selected = () => cells().filter(e => e.matches('[data-sm="primary"],[data-sm="secondary"]') && e.offsetParent !== null);
  const clearSelection = () => document.querySelectorAll('[data-sm="primary"],[data-sm="secondary"]').forEach(e => e.setAttribute('data-sm', 'unselected'));
  const choose = (e, direction = 1) => {
    if (!e) return;
    approach = direction;
    if (e.matches('.craft-omit-marker')) {
      clearSelection();
      e.focus({preventScroll: true});
      e.scrollIntoView({block: 'nearest'});
    } else {
      marker()?.blur();
      window.selectMsg(window.$(e));
      document.activeElement?.closest('.craft-omit-marker')?.blur();
    }
  };
  const restore = button => {
    const ids = JSON.parse(button.dataset.omitIds);
    ids.forEach(id => { omitted.delete(id); window.craftParts?.restoreCell(id); });
    const target = document.getElementById(approach < 0 ? ids.at(-1) : ids[0]);
    sync();
    if (target) choose(target);
    return true;
  };
  const sync = () => {
    frame = null;
    const all = cells(), runs = [];
    let run;
    for (const e of all) {
      const hide = omitted.has(e.id) || !!window.craftParts?.fullyHidden(e);
      if (e.classList.contains('craft-omitted') !== hide) e.classList.toggle('craft-omitted', hide);
      if (!eligible(e) || !hide) { run = null; continue; }
      if (!run) { run = []; runs.push(run); }
      run.push(e);
    }
    const keys = new Set();
    for (const group of runs) {
      const key = group[0].id, ids = group.map(e => e.id);
      keys.add(key);
      let button = markers.get(key);
      if (!button) {
        button = mountTpl('omit_marker');
        on(button, {click: () => restore(button), focus: clearSelection});
        markers.set(key, button);
      }
      const text = `▸ ${ids.length}`, label = `Expand ${ids.length} omitted ${ids.length === 1 ? 'cell' : 'cells'}`;
      if (button.textContent !== text) button.textContent = text;
      if (button.getAttribute('aria-label') !== label) {
        button.setAttribute('aria-label', label);
        button.title = label + ' · Right arrow, Enter, or Omit';
      }
      button.setAttribute('aria-expanded', 'false');
      button.dataset.omitIds = JSON.stringify(ids);
      if (button.nextElementSibling !== group[0]) group[0].before(button);
    }
    for (const [key, button] of markers) {
      if (keys.has(key)) continue;
      const focused = document.activeElement === button;
      const ids = JSON.parse(button.dataset.omitIds);
      button.remove();
      markers.delete(key);
      if (focused) {
        const replacement = [...markers.values()].find(b => JSON.parse(b.dataset.omitIds).some(id => ids.includes(id)));
        replacement?.focus();
      }
    }
    window.craftCellChains?.sync();
  };
  const toggle = () => {
    if (blocked()) return false;
    if (marker()) return restore(marker());
    const targets = selected();
    if (!targets.length) return false;
    const editing = document.getElementById('id_')?.value;
    if (targets.some(e => e.dataset.editing === 'true' || e.id === '_' + editing)) {
      window.UIkit?.notification('Finish editing the cell before omitting it.', {status: 'warning'});
      return true;
    }
    targets.forEach(e => omitted.add(e.id));
    sync();
    const button = [...markers.values()].find(b => JSON.parse(b.dataset.omitIds).includes(targets[0].id));
    choose(button);
    return true;
  };
  const stop = e => { e.preventDefault(); e.stopImmediatePropagation(); };
  const keys = e => {
    if (blocked() || e.isComposing) return;
    const current = marker();
    if (!e.metaKey && !e.ctrlKey && !e.altKey && !e.shiftKey) {
      if (current && ['ArrowRight', 'Enter', ' '].includes(e.key)) { stop(e); restore(current); return; }
      const direction = ['ArrowDown','j'].includes(e.key) ? 1 : ['ArrowUp','k'].includes(e.key) ? -1 : 0;
      if (direction) {
        const visible = [...(root()?.children || [])].filter(n => n.matches('.editable,.craft-omit-marker') && n.offsetParent !== null);
        const active = current || document.querySelector('#dialog-container > [data-sm="primary"]');
        const index = visible.indexOf(active), next = visible[index + direction];
        if (current || next?.matches('.craft-omit-marker')) { stop(e); choose(next, direction); return; }
      }
      if (current && ['Home','End'].includes(e.key)) {
        const visible = [...root().children].filter(n => n.matches('.editable,.craft-omit-marker') && n.offsetParent !== null);
        stop(e); choose(e.key === 'Home' ? visible[0] : visible.at(-1)); return;
      }
    }
    if (current && e.key !== 'Tab') stop(e);
  };
  const pointer = e => { if (marker() && !e.target.closest('.craft-omit-marker')) marker().blur(); };
  const ac = new AbortController();
  const offDom = onDomChange(sync, {attrs: ['class', 'hidden', 'data-sm-header'], delay: 'raf', inDialog: true});
  on(document, {keydown: keys, pointerdown: pointer}, {capture: true, signal: ac.signal});
  window.__hotkeyRegister?.({id: 'cell-omit', label: 'Toggle omit cells', group: 'Dialog · Presentation', combo: 'u', handler: () => window.craftOmit?.toggle() === true});
  window.craftOmit = {
    toggle, sync,
    disable() {
      offDom();
      ac.abort();
      markers.forEach(b => b.remove());
      cells().forEach(e => e.classList.remove('craft-omitted'));
      window.craftCellChains?.sync();
    }
  };
  sync();
})();


(() => {
  window.craftParts?.disable();
  const states = new Map(), controls = new Map(), pending = new Map();
  const originals = window.solveitHotkeys?.list().filter(s => ['si-i','si-o'].includes(s.id)) || [];
  let bypass = false, frame;
  const {qsa, on, onDomChange} = window.__craftCMCore;
  const cells = () => qsa('#dialog-container > .editable');
  const card = (e, side) => e.querySelector(side === 'input' ? ':scope > .card-in' : ':scope > .card-out');
  const button = (e, side) => card(e, side)?.querySelector('[hx-post="/collapse_"]');
  const present = (e, side) => side === 'input' ? !!card(e, side) : !!card(e, side)?.querySelector('.msg-body');
  const blocked = () => window.__craftCMCore.typingBlocked();
  const busy = e => e.dataset.editing === 'true' || e.id === '_' + document.getElementById('id_')?.value;
  const waiting = (e, side) => {
    const key = e.id + side, p = pending.get(key);
    if (p && (p.collapsed !== !!card(e, side)?.classList.contains('collapsed') || performance.now() > p.until)) pending.delete(key);
    return pending.has(key);
  };
  const native = (e, side) => {
    const b = button(e, side);
    if (!b || waiting(e, side)) return;
    const selection = cells().map(n => [n, n.getAttribute('data-sm')]);
    window.selectMsg($(e), {scroll: false});
    pending.set(e.id + side, {collapsed: !!card(e, side)?.classList.contains('collapsed'), until: performance.now() + 5000});
    bypass = true;
    try { b.click(); }
    finally {
      bypass = false;
      selection.forEach(([n, value]) => value === null ? n.removeAttribute('data-sm') : n.setAttribute('data-sm', value));
    }
  };
  const restore = (id, side) => {
    const state = states.get(id), e = document.getElementById(id);
    if (!state || !e) return;
    state.hidden.delete(side);
    if (!state.hidden.size) states.delete(id);
    sync();
    if (card(e, side)?.classList.contains('collapsed')) native(e, side);
    window.selectMsg($(e), {scroll: false});
  };
  const restoreCell = id => {
    const state = states.get(id);
    if (state) restore(id, state.last);
  };
  const fullyHidden = e => {
    const state = states.get(e.id);
    return !!state && ['input','output'].filter(s => present(e, s)).every(s => state.hidden.has(s));
  };
  const focusMarker = id => {
    const b = [...document.querySelectorAll('.craft-omit-marker')].find(n => JSON.parse(n.dataset.omitIds).includes(id));
    b?.focus({preventScroll: true});
  };
  const sync = () => {
    frame = null;
    const live = new Set();
    for (const e of cells()) {
      const state = states.get(e.id), full = fullyHidden(e);
      for (const side of ['input','output']) {
        const c = card(e, side), hidden = !!state?.hidden.has(side), key = e.id + ':' + side;
        if (c && c.classList.contains('craft-part-hidden') !== hidden) c.classList.toggle('craft-part-hidden', hidden);
        if (!hidden || !c || full) continue;
        live.add(key);
        let b = controls.get(key);
        if (!b) {
          b = window.__craftCMCore.el('button', {
            type: 'button', cls: 'craft-part-restore', title: `Restore ${side}`,
            'aria-label': `Restore ${side}`, onClick: () => restore(e.id, side),
          }, side === 'input' ? '⌃' : '⌄');
          controls.set(key, b);
        }
        if (side === 'input') {
          const output = card(e, 'output');
          if (output && b.nextElementSibling !== output) output.before(b);
        } else if (b.previousElementSibling !== card(e, 'input')) card(e, 'input')?.after(b);
      }
    }
    for (const [key, b] of controls) if (!live.has(key)) { b.remove(); controls.delete(key); }
    window.craftOmit?.sync();
  };
  const cycleOne = (e, side) => {
    if (!present(e, side) || waiting(e, side)) return;
    if (busy(e)) {
      window.UIkit?.notification('Finish editing before minimizing this cell.', {status: 'warning'});
      return;
    }
    const state = states.get(e.id);
    if (state?.hidden.has(side)) { restore(e.id, side); return; }
    if (!card(e, side).classList.contains('collapsed')) { native(e, side); return; }
    const next = state || {hidden: new Set(), last: side};
    next.hidden.add(side);
    next.last = side;
    states.set(e.id, next);
    sync();
    if (fullyHidden(e)) focusMarker(e.id);
  };
  const cycle = side => {
    if (blocked()) return false;
    const marker = document.activeElement?.closest('.craft-omit-marker');
    if (marker) {
      for (const id of JSON.parse(marker.dataset.omitIds)) if (states.get(id)?.hidden.has(side)) restore(id, side);
      return true;
    }
    const selected = cells().filter(e => e.matches('[data-sm="primary"],[data-sm="secondary"]') && e.offsetParent !== null);
    selected.forEach(e => cycleOne(e, side));
    return !!selected.length;
  };
  const click = event => {
    if (bypass) return;
    const b = event.target.closest('[hx-post="/collapse_"]');
    const c = b?.closest('.msg-card'), e = c?.closest('.editable');
    if (!e) return;
    event.preventDefault();
    event.stopImmediatePropagation();
    cycleOne(e, c.classList.contains('card-in') ? 'input' : 'output');
  };
  const finished = event => {
    const b = event.detail?.elt?.closest?.('[hx-post="/collapse_"]');
    const c = b?.closest('.msg-card'), e = c?.closest('.editable');
    if (e) pending.delete(e.id + (c.classList.contains('card-in') ? 'input' : 'output'));
  };
  const ac = new AbortController();
  const offDom = onDomChange(sync, {attrs: ['class'], delay: 'raf', inDialog: true});
  on(document, 'click', click, {capture: true, signal: ac.signal});
  on(document, 'htmx:afterRequest', finished, {signal: ac.signal});
  window.__hotkeyRegister?.({id: 'si-i', label: 'Cycle input: expanded / collapsed / hidden', group: 'Solveit · Dialog (selection mode)', combo: 'i', handler: () => cycle('input')});
  window.__hotkeyRegister?.({id: 'si-o', label: 'Cycle output: expanded / collapsed / hidden', group: 'Solveit · Dialog (selection mode)', combo: 'o', handler: () => cycle('output')});
  window.craftParts = {
    cycle, sync, fullyHidden, restoreCell,
    disable() {
      offDom();
      ac.abort();
      controls.forEach(b => b.remove());
      states.clear();
      qsa('.craft-part-hidden').forEach(c => c.classList.remove('craft-part-hidden'));
      originals.forEach(s => window.__hotkeyRegister?.({...s, handler: () => true}));
      window.craftOmit?.sync();
    }
  };
  sync();
})();


(() => {
  window.craftInsertPreview?.disable();
  if (!document.getElementById('dialog-container')) return;
  const types = {Code: 1, Note: 1, Raw: 1, Prompt: 1};
  const icon = name => window.__craftTpl?.[`cell_icon_${name}`] || '';
  const key = 'craft-insert-preview:v1:' + (new URLSearchParams(location.search).get('name') || location.pathname);
  const {mountTpl, qs, qsa, on, store, patch, onDomChange} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const trigger = mountTpl('insert_trigger'), menu = mountTpl('insert_menu');
  let current;
  const select = name => {
    current = name;
    const title = `A/B inserts: ${name}`;
    patch(trigger, {innerHTML: icon(name), title, 'aria-label': title});
    for (const option of qsa('button', menu)) option.setAttribute('aria-pressed', String(option.dataset.kind === name));
  };
  const saved = store.get(key);
  select(Object.hasOwn(types, saved) ? saved : document.getElementById('default_code')?.checked ? 'Code' : 'Prompt');
  let scroller;
  const position = () => {
    if (!scroller) return;
    const rect = scroller.getBoundingClientRect();
    trigger.hidden = !rect.width || !rect.height;
    Object.assign(trigger.style, {left: 'auto', right: '42px', top: '6px'});
    Object.assign(menu.style, {right: '10px', top: '36px'});
    window.craftMobile?.reposition?.();
  };
  const resize = new ResizeObserver(position);
  const sync = () => {
    const next = document.getElementById('dialog-container');
    if (scroller === next) return;
    if (scroller) {
      resize.unobserve(scroller);
      scroller.classList.remove('craft-insert-preview');
    }
    scroller = next;
    if (!scroller) { trigger.hidden = true; return; }
    scroller.classList.add('craft-insert-preview');
    resize.observe(scroller); position();
  };
  const choose = name => {
    select(name);
    store.set(key, name);
  };
  const activeCells = () => qsa('.editable:is([data-sm="primary"],[data-sm="secondary"])', scroller).filter(el => el.getClientRects().length);
  const cellType = el => el?.querySelector('.card-in[data-mtype]')?.dataset.mtype;
  const editorCell = () => (document.querySelector('#full_editor #id_')?.value || '').replace(/^_/, '');
  const notify = error => {
    console.error('CRAFT insertion:', error);
    window.UIkit?.notification?.({message: error.message, status: 'danger'});
  };
  const waitFor = (check, timeout = 5000) => {
    const immediate = check();
    if (immediate) return Promise.resolve(immediate);
    return new Promise(resolve => {
      let done = false;
      const finish = val => {
        if (done) return;
        done = true;
        offDomWait();
        clearInterval(tick);
        clearTimeout(cap);
        resolve(val);
      };
      const probe = () => { const r = check(); if (r) finish(r); };
      const offDomWait = onDomChange(probe, {attrs: ['data-mtype', 'class', 'hidden', 'data-editing'], delay: 0});
      const tick = setInterval(probe, 100);
      const cap = setTimeout(() => finish(null), timeout);
    });
  };
  const setType = async (id, type) => {
    if (cellType(document.getElementById(id)) !== type) {
      await window.__craftCMCore.htmxPost('/set_mtyp_', {id_: id, ids: id, mtyp: type});
      if (!(await waitFor(() => cellType(document.getElementById(id)) === type))) throw new Error(`Could not switch cell to ${type}.`);
    }
  };
  const blocked = () => window.__craftCMCore.typingBlocked('.cm-editor,#craft-insert-preview-menu');
  let busy = false;
  const execute = action => {
    if (!busy) {
      busy = true;
      Promise.resolve().then(action).catch(notify).finally(() => { busy = false; });
    }
    return true;
  };
  const mode = name => {
    if (blocked()) return false;
    choose(name);
    const cells = activeCells();
    if (cells.length !== 1) return true;
    return execute(async () => { await setType(cells[0].id, name.toLowerCase()); });
  };
  // opts.type ('Code'|'Note'|'Prompt'|'Raw') overrides the picker; opts.cell targets a cell other than the
  // selection; opts.force skips the focus guard (the mobile toolbar has already closed any editor).
  const insert = (where, opts = {}) => {
    if (!opts.force && blocked()) return false;
    const primary = opts.cell || activeCells().find(el => el.dataset.sm === 'primary');
    const button = primary?.querySelector(`[hx-post="/add_${where}_"]`);
    if (!button) return true;
    const name = Object.hasOwn(types, opts.type || '') ? opts.type : current;
    return execute(async () => {
      const type = name.toLowerCase(), old = new Set([...scroller.querySelectorAll('.editable[id]')].map(el => el.id));
      if (!window.craftC2?.guardNonCodeInsertion) throw new Error('Reload this dialog to enable typed insertion.');
      if (type === 'code') {
        if (!window.craftC2?.expectCodeInsertion) throw new Error('Reload this dialog to enable C2 insertion.');
        window.craftC2.expectCodeInsertion();
      }
      const release = type !== 'code' ? window.craftC2.guardNonCodeInsertion(old) : null;
      const unhide = window.craftIdleComposer?.holdForInsertion();
      const ph = window.__craftCMCore.mountTpl('insert_placeholder');
      if (where === 'above') primary.before(ph); else primary.after(ph);
      let id = null;
      try {
        button.dispatchEvent(new MouseEvent('click', {bubbles:false}));
        id = await waitFor(() => {
          const editing = editorCell(), id = '_' + editing;
          if (editing && !old.has(id) && document.getElementById(id)) return id;
          if (type === 'code') return [...scroller.querySelectorAll('.editable[id]')].find(el => !old.has(el.id) && window.craftC2?.manages(el.id))?.id;
        });
        if (!id) throw new Error(`Could not confirm ${name} insertion.`);
        if (type === 'code') {
          if (!(await waitFor(() => cellType(document.getElementById(id)) === 'code' && editorCell() !== id.slice(1) && Boolean(document.getElementById(id)?.querySelector('.craft-c2-host'))))) throw new Error('Could not confirm C2 handoff.');
        } else {
          await setType(id, type);
          const cancel = await waitFor(() => {
            if (editorCell() !== id.slice(1) || window.edValue?.()) return true;
            const btn = document.getElementById('cancel_btn');
            return btn && !btn.disabled ? btn : null;
          });
          if (!cancel) throw new Error('Could not close the insertion composer.');
          if (cancel && cancel !== true) {
            const field = document.querySelector('#full_editor #msg_type');
            if (field) field.value = type;
            cancel.click();
            if (!(await waitFor(() => editorCell() !== id.slice(1)))) throw new Error('Could not close the insertion composer.');
            if (cellType(document.getElementById(id)) !== type) throw new Error(`New cell did not retain type ${type}.`);
          }
        }
      } finally { ph.remove(); release?.(); unhide?.(); }
      if (id) {
        document.getElementById(id)?.scrollIntoView({block: 'nearest'});
        if (type === 'code') {
          if (!document.activeElement?.closest?.(`#${id} .craft-c2-host`)) window.craftC2?.edit?.(id, {anchor: 0, head: 0});
        } else if (!document.activeElement?.closest?.(`#${id} .craft-md-host`)) {
          window.craftMd?.edit?.(id, {toEnd: true});
        }
      }
    });
  };
  const defs = [
    ['si-a', 'a', 'Add message above', () => insert('above')],
    ['si-b', 'b', 'Add message below', () => insert('below')],
    ...['Code','Note','Prompt','Raw'].map((name,i) => [`si-mode-${name.toLowerCase()}`, `Mod+Shift+${'JKL;'[i]}`, `Switch message to ${name.toLowerCase()}`, () => mode(name)]),
  ].map(([id,combo,label,handler]) => ({id,combo,label,handler,group:'Solveit · Dialog (selection mode)'}));
  const registerBindings = () => defs.forEach(spec => window.__hotkeyRegister?.(spec));
  const builtins = qs('script[data-solveit-hotkeys="builtins"]');
  on(builtins, 'load', registerBindings, {once: true, signal: ac.signal});
  registerBindings();
  on(menu, 'click', e => {
    const option = e.target.closest('button[data-kind]');
    if (!option) return;
    choose(option.dataset.kind);
    menu.hidePopover(); trigger.focus();
  }, sig);
  document.body.append(trigger, menu);
  const KIND_MAP = {code: 'Code', note: 'Note', prompt: 'Prompt', raw: 'Raw'};
  const onHtmxConfig = e => {
    const path = e.detail?.path || '';
    const mtyp = e.detail?.parameters?.mtyp || e.detail?.parameters?.msg_type;
    if ((path.includes('set_mtyp') || path.includes('upsert_msg')) && mtyp && KIND_MAP[String(mtyp).toLowerCase()]) {
      choose(KIND_MAP[String(mtyp).toLowerCase()]);
    }
  };
  const onTabClick = e => {
    const tab = e.target?.closest?.('#full_editor [role="tab"], #full_editor button');
    const name = tab && KIND_MAP[(tab.textContent || '').trim().toLowerCase()];
    if (name) choose(name);
  };
  on(document, 'htmx:configRequest', onHtmxConfig, sig);
  on(document, 'click', onTabClick, cap);
  const offDom = onDomChange(sync, {delay: 0});
  on(window, 'resize', position, sig);
  window.craftInsertPreview = {
    insert: (where, opts) => insert(where, opts),
    choose,
    setType,
    type: () => current,
    isBusy: () => busy,
    disable() {
    ac.abort();
    offDom(); resize.disconnect();
    defs.forEach(spec => window.__hotkeyRegister?.({...spec, builtin:true, handler:() => true}));
    scroller?.classList.remove('craft-insert-preview');
    trigger.remove(); menu.remove(); delete window.craftInsertPreview;
  }};
  sync();
})();


(() => {
  window.craftInlineNote?.disable?.();
  window.craftInlineText?.disable?.();
  if (!document.getElementById('dialog-container')) return;
  if (!document.querySelector('script[data-craft-md="1"]')) {
    const s = document.createElement('script');
    s.src = '/static/.craft-assets/craft_md.js';
    s.dataset.craftMd = '1';
    document.head.append(s);
  }
})();

(() => {
  window.craftImageDrop?.disable();
  const {qs, on} = window.__craftCMCore;
  const target = e => {
    if (e.target.closest?.('.monaco-editor')) return;
    return e.target.closest?.('#dialog-container > .editable > .card-in:is([data-mtype="code"],[data-mtype="note"],[data-mtype="prompt"])');
  };
  const field = id => qs(`#full_editor #${id}`)?.value;
  const editorId = () => (field('id_') || '').replace(/^_/, '');
  const model = () => window.editor?.getModel();
  const notify = message => window.UIkit?.notification?.({message, status: 'warning'});
  let busy = false, disabled = false;
  const dragover = e => {
    if (!target(e)) return;
    const items = [...(e.dataTransfer?.items || [])];
    if (!items.some(i => i.kind === 'file' && (!i.type || i.type.startsWith('image/'))) && (items.length || !e.dataTransfer?.types.includes('Files'))) return;
    e.preventDefault();
    e.dataTransfer.dropEffect = 'copy';
  };
  const attach = async (e, card, file) => {
    if (typeof window.handleBlob !== 'function') return;
    e.preventDefault();
    e.stopPropagation();
    if (busy) return notify('Wait for the current image upload to finish.');
    const cell = card.parentElement, id = cell.id.slice(1), kind = card.dataset.mtype;
    const priorModel = model();
    const ready = () => editorId() === id && field('is_input') === '1' && field('msg_type') === kind && model() && !model().uri.path.endsWith('/temp');
    busy = true;
    try {
      if (!ready()) {
        if (editorId() || window.edValue?.()) return notify('Finish editing the open cell before attaching an image to another cell.');
        if (window.craftC2?.active?.() === id) await window.craftC2.exit(id);
        if (window.craftMd?.active?.() === `${id}:in`) await window.craftMd.exit(id, 'in');
        const btn = card.querySelector('[hx-post="/editor_"]');
        if (btn) { btn.dataset.craftAllowMonaco = '1'; btn.click(); delete btn.dataset.craftAllowMonaco; }
        for (let i = 0; i < 100 && !disabled && !(ready() && model() !== priorModel); i++) await new Promise(resolve => setTimeout(resolve, 50));
        if (disabled) return;
        if (!ready() || model() === priorModel) return notify('Could not open the cell to attach the image.');
      }
      window.craftIdleComposer?.reveal();
      window.editor.focus();
      const ext = file.name.match(/\.([^.]+)$/)?.[1] || file.type.split('/')[1] || 'png';
      await window.handleBlob(file, ext, true, file.name);
    } catch (err) {
      console.error('CRAFT image attachment:', err);
      notify('Could not attach the image.');
    } finally { busy = false; }
  };
  const drop = e => {
    const card = target(e), file = e.dataTransfer?.files[0];
    if (card && file?.type.startsWith('image/')) attach(e, card, file);
  };
  const ac = new AbortController();
  on(window, {dragover, drop}, {capture: true, signal: ac.signal});
  window.craftImageDrop = {disable() {
    disabled = true;
    ac.abort();
    delete window.craftImageDrop;
  }};
})();

/* ── Progressive table enhancer (wrap, numeric align, flat-table sort, copy TSV) ── */
(() => {
  const NUM_RE = /^[-+\u2212]?[$€£¥]?\s*\d[\d,_]*(?:\.\d+)?(?:[eE][+-]?\d+)?%?$/;
  const EMPTY_RE = /^(?:|nan|na|n\/a|null|none|-|\u2014)$/i;
  const parseNum = s => {
    const t = s.trim();
    if (!t || EMPTY_RE.test(t)) return null;
    if (!NUM_RE.test(t)) return NaN;
    return parseFloat(t.replace(/[\u2212]/g, '-').replace(/[$€£¥,\s_%]/g, ''));
  };

  const enhanceTable = tbl => {
    if (!tbl || tbl.dataset.craftTable) return;
    tbl.dataset.craftTable = '1';

    // 1. Wrap in a keyboard-scrollable container if not already wrapped
    let wrap = tbl.parentElement;
    if (!wrap?.classList.contains('craft-table-wrap')) {
      wrap = window.__craftCMCore.mountTpl('table_wrap');
      tbl.parentNode.insertBefore(wrap, tbl);
      wrap.appendChild(tbl);
    }

    const headRows = [...tbl.querySelectorAll(':scope > thead > tr')];
    const tbody = tbl.querySelector(':scope > tbody');
    const bodyRows = tbody ? [...tbody.querySelectorAll(':scope > tr')] : [];
    const hasSpans = [...tbl.querySelectorAll('th, td')].some(c => c.colSpan > 1 || c.rowSpan > 1);

    // 2. Detect numeric columns and right-align both header and cells
    if (!hasSpans && bodyRows.length) {
      const sample = bodyRows.slice(0, 40);
      const colCount = sample[0].children.length;
      const headCells = headRows.length === 1 ? [...headRows[0].children] : null;
      for (let col = 0; col < colCount; col++) {
        let numHits = 0, nonEmpty = 0;
        for (const row of sample) {
          const cell = row.children[col];
          if (!cell || cell.tagName === 'TH') continue;
          const v = parseNum(cell.textContent || '');
          if (v === null) continue;
          nonEmpty++;
          if (!Number.isNaN(v)) numHits++;
        }
        if (nonEmpty > 0 && numHits === nonEmpty) {
          for (const row of bodyRows) row.children[col]?.classList.add('craft-num');
          if (headCells && headCells[col]) headCells[col].classList.add('craft-num');
        }
      }
    } else if (hasSpans && bodyRows.length) {
      // MultiIndex: align numeric <td> cells (and matching top-row headers) without disturbing rowspan
      for (const row of bodyRows) {
        for (const cell of row.children) {
          if (cell.tagName !== 'TD') continue;
          const v = parseNum(cell.textContent || '');
          if (v !== null && !Number.isNaN(v)) cell.classList.add('craft-num');
        }
      }
      if (headRows.length && headRows[0].children.length === bodyRows[0].children.length) {
        for (let col = 0; col < bodyRows[0].children.length; col++) {
          if (bodyRows[0].children[col]?.classList.contains('craft-num')) {
            headRows[0].children[col]?.classList.add('craft-num');
          }
        }
      }
    }

    // 3. Click/keyboard sort on flat tables (>= 2 rows, single header row, no spans)
    if (!hasSpans && headRows.length === 1 && bodyRows.length >= 2) {
      const origOrder = bodyRows.slice();
      const ths = [...headRows[0].children];
      ths.forEach((th, colIdx) => {
        if (!(th.textContent || '').trim()) return;
        th.classList.add('craft-sort');
        th.tabIndex = 0;
        th.setAttribute('aria-sort', 'none');
        const toggle = () => {
          const cur = th.getAttribute('aria-sort');
          const next = cur === 'ascending' ? 'descending' : cur === 'descending' ? 'none' : 'ascending';
          for (const other of ths) if (other !== th && other.classList.contains('craft-sort')) other.setAttribute('aria-sort', 'none');
          th.setAttribute('aria-sort', next);
          if (next === 'none') {
            tbody.append(...origOrder);
            return;
          }
          const dir = next === 'ascending' ? 1 : -1;
          const isNum = th.classList.contains('craft-num');
          const sorted = origOrder.slice().sort((a, b) => {
            const ta = (a.children[colIdx]?.textContent || '').trim();
            const tb = (b.children[colIdx]?.textContent || '').trim();
            if (isNum) {
              const na = parseNum(ta), nb = parseNum(tb);
              const va = (na === null || Number.isNaN(na)) ? Infinity : na;
              const vb = (nb === null || Number.isNaN(nb)) ? Infinity : nb;
              return (va - vb) * dir;
            }
            return ta.localeCompare(tb, undefined, { numeric: true, sensitivity: 'base' }) * dir;
          });
          tbody.append(...sorted);
        };
        th.addEventListener('click', toggle);
        th.addEventListener('keydown', e => {
          if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); toggle(); }
        });
      });
    }

    // 4. Subtle Copy TSV button on output tables
    if (tbl.closest('.card-out') && !wrap.querySelector(':scope > .craft-table-copy')) {
      const btn = window.__craftCMCore.mountTpl('table_copy_btn');
      btn.addEventListener('click', async e => {
        e.stopPropagation();
        const rows = [...tbl.querySelectorAll('tr')].map(tr =>
          [...tr.children].map(c => (c.textContent || '').replace(/[\t\r\n]+/g, ' ').trim()).join('\t')
        ).join('\n');
        try {
          await navigator.clipboard.writeText(rows);
          btn.textContent = 'Copied';
          btn.classList.add('is-copied');
          setTimeout(() => { btn.textContent = 'Copy TSV'; btn.classList.remove('is-copied'); }, 1400);
        } catch {}
      });
      wrap.insertBefore(btn, tbl);
    }
  };

  const enhanceAll = () => {
    for (const tbl of document.querySelectorAll('#dialog-container .msg-content table:not([data-craft-table])')) {
      enhanceTable(tbl);
    }
  };
  const start = () => {
    window.__craftCMCore.onDomChange(enhanceAll, {delay: 80, inDialog: true});
    enhanceAll();
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', start);
  else start();
  window.craftTables = { enhance: enhanceAll, disable() { mo.disconnect(); clearTimeout(timer); } };
})();

/* ── Accessibility: mirror uk-tooltip & fallback aria-labels on icon controls ── */
(() => {
  const extractTip = raw => {
    if (!raw) return '';
    const s = String(raw).trim();
    const m = s.match(/(?:^|;)\s*title\s*:\s*([^;]+)/i);
    return (m ? m[1] : s).replace(/<[^>]*>/g, '').trim();
  };
  const labelControls = (root = document) => {
    // 1. Any interactive element carrying uk-tooltip / data-uk-tooltip without an accessible name
    for (const el of root.querySelectorAll('[uk-tooltip], [data-uk-tooltip]')) {
      if (!el.matches('button, a, input, select, textarea')) continue;
      if (el.hasAttribute('aria-label') || el.hasAttribute('aria-labelledby')) continue;
      const tip = extractTip(el.getAttribute('uk-tooltip') || el.getAttribute('data-uk-tooltip'));
      if (tip) el.setAttribute('aria-label', tip);
    }
    // 2. #find-box controls (checkboxes wrapped in <label uk-tooltip>, search/replace inputs)
    const fb = root.getElementById ? root.getElementById('find-box') : root.querySelector?.('#find-box');
    if (fb) {
      for (const el of fb.querySelectorAll('input, button, select')) {
        if (el.hasAttribute('aria-label') || el.hasAttribute('aria-labelledby')) continue;
        const lbl = el.closest('label') || (el.id && fb.querySelector(`label[for="${CSS.escape(el.id)}"]`));
        const tip = extractTip(
          el.getAttribute('uk-tooltip') || el.getAttribute('data-uk-tooltip') ||
          lbl?.getAttribute('uk-tooltip') || lbl?.getAttribute('data-uk-tooltip') ||
          el.getAttribute('title') || lbl?.getAttribute('title')
        );
        const fallback = tip || (el.getAttribute('placeholder') || '').replace(/\.+$/, '').trim();
        if (fallback) el.setAttribute('aria-label', fallback);
      }
    }
    // 3. Navbar <select name="mode"> and Dialogs link
    const nav = root.querySelector?.('.uk-container > nav.monster-navbar') || document.querySelector('.uk-container > nav.monster-navbar');
    if (nav) {
      const modeSel = nav.querySelector('select[name="mode"]');
      if (modeSel && !modeSel.hasAttribute('aria-label')) modeSel.setAttribute('aria-label', 'AI mode');
      const dlgsLink = nav.querySelector('a[href^="/?at="]');
      if (dlgsLink && !dlgsLink.hasAttribute('aria-label') && !(dlgsLink.textContent || '').trim()) {
        dlgsLink.setAttribute('aria-label', 'Dialogs');
      }
    }
  };
  let offDom = null;
  const start = () => {
    labelControls(document);
    offDom = window.__craftCMCore.onDomChange(() => labelControls(document), {delay: 120});
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', start);
  else start();
  window.craftA11y = { refresh: () => labelControls(document), disable() { offDom?.(); } };
})();

/* ── Desktop UX: hover-edge drag-to-insert, click-margin escape, Shift+A/B seam picker & slash-morph ── */
(() => {
  window.craftDesktopUX?.disable?.();
  if (!document.getElementById('dialog-container')) return;
  const {mountTpl, on} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const isMobile = () => !!window.craftMobile?.active?.();
  const dialog = () => document.getElementById('dialog-container');
  const CELL_TYPES = window.__craftCfg?.cell_types || ['Code', 'Note', 'Prompt', 'Raw'];
  const KEY_TO_TYPE = window.__craftCfg?.key_to_type || {c: 'Code', n: 'Note', p: 'Prompt', r: 'Raw'};
  const SLASH_RE = /^\/(c|code|n|note|p|prompt|r|raw)[ \t\n]$/i;
  const SLASH_MAP = window.__craftCfg?.slash_map || {c: 'Code', code: 'Code', n: 'Note', note: 'Note', p: 'Prompt', prompt: 'Prompt', r: 'Raw', raw: 'Raw'};
  /* Bottom hover-edge + handle on desktop (shown when hovering near bottom-center) */
  const hBelow = mountTpl('desk_edge_below');
  const syncHandleKinds = () => {
    const t = window.craftInsertPreview?.type?.() || 'Code';
    const k = hBelow.querySelector('.craft-desk-edge-kind');
    if (k) k.innerHTML = window.__craftTpl?.[`desk_icon_${t}`] || window.__craftTpl?.desk_icon_Code || '';
    hBelow.title = `Click to insert ${t} below · Drag to choose type`;
  };

  /* Shared 4-option fan picker for both drag-to-insert and Shift+A/B */
  const picker = mountTpl('desk_picker');
  document.body.append(picker);

  let hostCell = null, pState = null; // {edge, cell, pointerId, startX, startY, dragged, hotType, keyMode}
  const detachHandles = () => {
    hBelow.removeAttribute('data-near');
    hostCell?.classList.remove('craft-desk-host');
    hBelow.remove();
    hostCell = null;
  };
  const attachHandles = cell => {
    if (isMobile() || !cell?.isConnected) { detachHandles(); return; }
    if (cell === hostCell) { syncHandleKinds(); return; }
    detachHandles();
    hostCell = cell;
    cell.classList.add('craft-desk-host');
    syncHandleKinds();
    cell.append(hBelow);
  };
  const setHot = type => {
    if (pState) pState.hotType = type;
    for (const b of picker.querySelectorAll('button[data-type]')) {
      b.toggleAttribute('data-hot', b.dataset.type === type);
    }
  };
  const closePicker = () => {
    hBelow.removeAttribute('data-active');
    picker.classList.remove('open');
    setHot(null);
    pState = null;
  };
  const positionPicker = (anchorRect, edge, hint) => {
    picker.querySelector('.craft-desk-picker-head').textContent = hint || (edge === 'above' ? 'Insert above' : 'Insert below');
    picker.classList.add('open');
    const pw = picker.offsetWidth || 270, ph = picker.offsetHeight || 68;
    const cx = anchorRect.left + anchorRect.width / 2;
    const left = Math.max(8, Math.min(cx - pw / 2, innerWidth - pw - 8));
    let top;
    if (edge === 'above') {
      const aboveTop = anchorRect.top - ph - 10;
      top = aboveTop >= 8 ? aboveTop : Math.min(anchorRect.bottom + 10, innerHeight - ph - 8);
    } else {
      const belowTop = anchorRect.bottom + 10;
      top = (belowTop + ph <= innerHeight - 8) ? belowTop : Math.max(8, anchorRect.top - ph - 10);
    }
    picker.style.left = Math.round(left) + 'px';
    picker.style.top = Math.round(top) + 'px';
  };
  const hitOpt = (x, y) => {
    const PAD = 8;
    for (const b of picker.querySelectorAll('button[data-type]')) {
      const r = b.getBoundingClientRect();
      if (x >= r.left - PAD && x <= r.right + PAD && y >= r.top - PAD && y <= r.bottom + PAD) return b.dataset.type;
    }
    return null;
  };
  async function commitEditors() {
    if (window.craftC2?.active?.()) await window.craftC2.exit();
    if (window.craftMd?.active?.()) await window.craftMd.exit();
  }
  async function doInsert(edge, cell, type) {
    if (type) window.craftInsertPreview?.choose?.(type);
    syncHandleKinds();
    await commitEditors();
    window.craftInsertPreview?.insert?.(edge, {type: type || window.craftInsertPreview?.type?.(), cell, force: true});
  }

  const onHandleDown = e => {
    if (isMobile() || e.button !== 0) return;
    const handle = e.currentTarget;
    const cell = handle.closest('#dialog-container > .editable[id]');
    if (!cell) return;
    e.preventDefault(); e.stopPropagation();
    closePicker();
    const edge = handle.dataset.edge;
    handle.dataset.active = 'true';
    handle.setPointerCapture?.(e.pointerId);
    pState = {edge, cell, pointerId: e.pointerId, startX: e.clientX, startY: e.clientY, dragged: false, hotType: null, keyMode: false};
  };
  const onHandleMove = e => {
    if (!pState || e.pointerId !== pState.pointerId || pState.keyMode) return;
    const dist = Math.hypot(e.clientX - pState.startX, e.clientY - pState.startY);
    if (!pState.dragged && dist > 6) {
      pState.dragged = true;
      positionPicker(e.currentTarget.getBoundingClientRect(), pState.edge, `Drop to insert ${pState.edge} & set default`);
    }
    if (pState.dragged) setHot(hitOpt(e.clientX, e.clientY));
  };
  const onHandleUp = e => {
    if (!pState || e.pointerId !== pState.pointerId || pState.keyMode) return;
    const {edge, cell, dragged, hotType} = pState;
    closePicker();
    if (!dragged) doInsert(edge, cell, null);
    else if (hotType) doInsert(edge, cell, hotType);
  };
  const onHandleCancel = e => { if (pState && e.pointerId === pState.pointerId) closePicker(); };
  const onHandleClick = e => { e.preventDefault(); e.stopPropagation(); };
  on(hBelow, {pointerdown: onHandleDown, pointermove: onHandleMove, pointerup: onHandleUp, pointercancel: onHandleCancel, click: onHandleClick}, sig);
  const onPickerClick = e => {
    const b = e.target.closest('button[data-type]');
    if (!b || !pState) return;
    e.preventDefault(); e.stopPropagation();
    const {edge, cell} = pState;
    closePicker();
    doInsert(edge, cell, b.dataset.type);
  };
  on(picker, 'click', onPickerClick, sig);

  /* Track cursor proximity to the bottom-center target area of a cell on desktop */
  const onMouseMove = e => {
    if (isMobile() || pState) return;
    const t = e.target;
    if (t instanceof Element && hBelow.contains(t)) {
      hBelow.dataset.near = 'true';
      return;
    }
    const cell = t instanceof Element ? t.closest('#dialog-container > .editable[id]') : null;
    if (cell) {
      attachHandles(cell);
      if (hostCell === cell) {
        const r = cell.getBoundingClientRect();
        const nearY = Math.abs(e.clientY - r.bottom) <= 26;
        const nearX = Math.abs(e.clientX - (r.left + r.width / 2)) <= 140;
        hBelow.toggleAttribute('data-near', nearY && nearX);
      }
    } else if (hostCell) {
      const r = hostCell.getBoundingClientRect();
      const nearY = Math.abs(e.clientY - r.bottom) <= 26;
      const nearX = Math.abs(e.clientX - (r.left + r.width / 2)) <= 140;
      if (nearY && nearX) hBelow.dataset.near = 'true';
      else hBelow.removeAttribute('data-near');
    }
  };
  const onDialogLeave = () => { if (!pState) detachHandles(); };
  on(document, 'mousemove', onMouseMove, {passive: true, signal: ac.signal});
  on(dialog(), 'mouseleave', onDialogLeave, sig);

  /* Click-margin / click-header to escape active inline editors on desktop */
  const onDocMouseDown = e => {
    if (isMobile()) return;
    if (pState?.keyMode && !picker.contains(e.target)) closePicker();
    if (e.button !== 0) return;
    const hasEd = !!(window.craftC2?.active?.() || window.craftMd?.active?.());
    if (!hasEd) return;
    const t = e.target;
    if (!(t instanceof Element)) return;
    if (t.closest('#solveit-file-tree-panel, #solveit-temp-editor, #solveit-launcher, .craft-action-menu, #craft-desk-picker, .craft-desk-edge, .cm-editor, .cm-tooltip, .uk-modal, .uk-dropdown, .uk-drop, nav.monster-navbar')) return;
    const inDialog = t.closest('#dialog-container');
    if (!inDialog) return;
    const inCell = t.closest('#dialog-container > .editable[id]');
    const onHeaderBlank = inCell && t.closest('.uk-card-header') && !t.closest('button, a, input, select');
    if (!inCell || onHeaderBlank) {
      commitEditors();
    }
  };
  on(document, 'mousedown', onDocMouseDown, cap);

  /* Shift+A / Shift+B inline seam quick-picker in selection mode */
  const blocked = () => document.activeElement?.closest?.('input,textarea,select,[contenteditable],.monaco-editor,.cm-editor,#full_editor,.hk-overlay.open,#solveit-launcher.open,[role="dialog"],.uk-modal.uk-open');
  const onKeyDown = e => {
    if (isMobile() || e.isComposing) return;
    if (pState?.keyMode) {
      if (e.key === 'Escape') { e.preventDefault(); e.stopImmediatePropagation(); closePicker(); return; }
      const t = KEY_TO_TYPE[e.key.toLowerCase()];
      if (t && !e.metaKey && !e.ctrlKey && !e.altKey) {
        e.preventDefault(); e.stopImmediatePropagation();
        const {edge, cell} = pState;
        closePicker();
        doInsert(edge, cell, t);
        return;
      }
      return;
    }
    if (e.shiftKey && !e.metaKey && !e.ctrlKey && !e.altKey && (e.key === 'A' || e.key === 'B')) {
      if (blocked()) return;
      const cell = dialog()?.querySelector(':scope > .editable[data-sm="primary"]');
      if (!cell) return;
      e.preventDefault(); e.stopImmediatePropagation();
      const edge = e.key === 'A' ? 'above' : 'below';
      const r = cell.getBoundingClientRect();
      const seamRect = edge === 'above'
        ? {left: r.left, width: r.width, top: r.top, bottom: r.top}
        : {left: r.left, width: r.width, top: r.bottom, bottom: r.bottom};
      pState = {edge, cell, pointerId: null, startX: 0, startY: 0, dragged: false, hotType: null, keyMode: true};
      positionPicker(seamRect, edge, `Insert ${edge} · press C / N / P / R`);
    }
  };
  on(document, 'keydown', onKeyDown, cap);

  /* Slash-morph on empty cells: typing /c, /code, /n, /note, /p, /prompt, /r, /raw + Space/Enter morphs the cell */
  let morphing = false;
  const checkSlashMorph = () => {
    if (morphing || isMobile()) return;
    let id = null, view = null, isCode = false;
    const c2Id = window.craftC2?.active?.();
    if (c2Id) {
      view = window.craftC2.view(c2Id);
      if (view?.hasFocus) { id = '_' + String(c2Id).replace(/^_/, ''); isCode = true; }
    }
    if (!id) {
      const mdKey = window.craftMd?.active?.();
      if (mdKey && mdKey.endsWith(':in')) {
        view = window.craftMd.view(mdKey);
        if (view?.hasFocus) id = '_' + mdKey.split(':')[0].replace(/^_/, '');
      }
    }
    if (!id || !view) return;
    const text = view.state.doc.toString();
    const m = SLASH_RE.exec(text);
    if (!m) return;
    const targetName = SLASH_MAP[m[1].toLowerCase()];
    const targetType = targetName.toLowerCase();
    const el = document.getElementById(id);
    const curType = el?.querySelector('.card-in[data-mtype]')?.dataset.mtype;
    if (!el || curType === targetType) return;
    morphing = true;
    view.dispatch({changes: {from: 0, to: text.length, insert: ''}});
    (async () => {
      try {
        await commitEditors();
        window.craftInsertPreview?.choose?.(targetName);
        if (window.craftInsertPreview?.setType) {
          await window.craftInsertPreview.setType(id, targetType);
        }
        if (targetType === 'code') window.craftC2?.edit?.(id.slice(1), {anchor: 0, head: 0});
        else window.craftMd?.edit?.(id.slice(1), {toEnd: true});
      } catch (err) {
        console.error('craftDesktopUX: slash-morph failed', err);
      } finally { morphing = false; }
    })();
  };
  const onSlashInput = () => { queueMicrotask(checkSlashMorph); };
  on(document, 'input', onSlashInput, cap);

  const offMobile = window.craftMobile?.onChange?.(active => { if (active) { closePicker(); detachHandles(); } });
  window.craftDesktopUX = {
    disable() {
      offMobile?.(); closePicker(); detachHandles(); picker.remove();
      ac.abort();
      delete window.craftDesktopUX;
    },
  };
})();


## Core Theme, Layout, Top Nav, Disclosures, Typography & Launcher CSS

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* Reserve top-right space inside desktop top nav so #craft-insert-preview and #craft-mobile-toggle never overlap navbar buttons. */
html:not(.craft-mobile) .uk-container > nav.monster-navbar {
  padding-right: 78px !important;
  box-sizing: border-box;
}
/* ══ Solveit "borderless" restyle · Tokyo Night Pure ══════════════════
   Injected as a <style> tag for every dialog in this folder. RELOAD to apply.
   Intent: cells float on one continuous canvas, full width, minimal chrome,
   separated by spacing and faint tone steps rather than boxes.

   Palette, from themes/tokyo-night-pure-color-theme.json:
     #16161e   panel/titleBar/sideBar   page canvas, recessed wells
     #1f2030   lifted editor.background card surface, composer (#1a1b26 was ~1:1 vs canvas)
     #25273a   derived                  output step (see note below)
     #cfc9c2   editor.foreground        body text
     #cfc9c266 editorLineNumber.fg      muted metadata label
     #7aa2f7   selection hue            active toggle glyph
     #7dcfff   Tokyo Night cyan         links (#1779b2 was ~3.6:1 on cards)

   The output step is invented: no theme key means "a shade off the editor".
   editor.lineHighlightBackground #2b2e43 is the semantically exact choice but
   a big jump; #25273a sits a half-step above the #1f2030 card instead.

   NOT remapped: --color-background and friends. Tailwind bg-* utilities still
   resolve to the original greys, which is why the nav <select> arrived black.

   Companion files in this folder:
     CRAFT.js   — marks `.dot-prompt` wrappers, and drives the top-nav reveal
                  via a `.nav-shown` class. Section 7 only styles the states;
                  the policy lives in the JS.
     Both are injected per dialog. RELOAD after editing either.

   Gotcha that cost two probes: UIkit paints its button chips as `outline`
   (inset / outset / groove, rgba(255,255,255,0.1)), NOT border or box-shadow.
   `outline` is not part of the `border` shorthand, so border/background/shadow
   resets leave the boxes visible. Section 7 now resets it. Do NOT add a
   blanket outline reset outside the nav: the app's [data-editing] cursor ring
   is an outline, and Tailwind's ring-2 selection ring is a box-shadow.

   Probing method: filter by getBoundingClientRect() and print the rect, so
   screenshot pixels map straight to elements — scoping a probe by selector
   presumes the answer. Always test border AND background AND box-shadow AND
   outline. Note `nav.monster-navbar` is not unique: every .uk-card-header
   holds one, so an unscoped probe returns card-header noise; the `>` child
   combinator is what keeps the real rules correct.
   ──────────────────────────────────────────────────────────────────── */

/* ── 1. Canvas and scroller ─────────────────────────────────────────── */
html, body { background-color: #1f2030 !important; }

div.overflow-y-auto.border.px-3 {        /* the only scroller; was a grey panel */
  background: transparent !important;
  border-color: transparent !important;
}
div.overflow-y-auto { scrollbar-width: none; }
div.overflow-y-auto::-webkit-scrollbar { width: 0; height: 0; }
div.overflow-y-auto:hover { scrollbar-width: thin; }
div.overflow-y-auto:hover::-webkit-scrollbar { width: 8px; height: 8px; }

/* ── 2. Layout: full width, spacing carries the structure ───────────── */
.uk-container {
  max-width: none !important;
  width: 100% !important;
  padding-inline: 0 !important;
  padding-bottom: 0 !important;          /* composer sits flush to the viewport */
}
.msg-card.card-in, .msg-card.card-out {
  width: 100% !important;
  margin-left: 0 !important;             /* defeats ml-auto on outputs */
  margin-right: 0 !important;
}
.editable {
  margin-block: 0 4px !important;        /* compact dialog cell gap */
  gap: 0 !important;                     /* input and output are flush */
}
/* Keep the selection ring, but remove the outer green editing outline. */
.editable[data-editing] { outline: none !important; }
/* The green halo was never a border: it is the app's diff tint. Only the
   `added` state is neutralised; `deleted` (red) and `changed` (yellow) stand. */
div.editable[data-diff="added"] { background: transparent !important; }
.msg-body { padding: 2px 6px !important; }
.card-out .msg-body { padding-left: 14px !important; }  /* subliminal offset */
/* Type scale in one place; CRAFT.js reads --craft-code for Monaco editors too. */
:root { --craft-prose: 11px; --craft-prose-lh: 13px; --craft-code: 10px; --craft-code-lh: 1.2; }
.msg-content, .msg-content .prose { font-size: var(--craft-prose) !important; line-height: var(--craft-prose-lh) !important; }
.msg-content pre, .msg-content code { font-size: var(--craft-code) !important; line-height: 1 !important; }
.msg-content .prose p { margin: 0.15em 0 !important; }

/* ── 3. Surfaces and text ───────────────────────────────────────────── */
.msg-card { background-color: #1f2030 !important; }
.msg-card.card-out { background-color: #25273a !important; }

.msg-content, .msg-content .prose { color: #cfc9c2 !important; }
.msg-content a { color: #7dcfff !important; }


/* Metadata label ("Code:1,257") recedes from the body text */
.msg-card > .uk-card-header nav > div.grow,
.msg-card > .uk-card-header nav > div.grow * {
  color: #cfc9c2a8 !important;
}

/* One tone per card: input code is a plain PRE that paints its own grey */
.msg-card pre, .msg-card code { background: transparent !important; }

/* Except in prompts and notes, where fenced code sits in a recessed square well.
   Keep background & padding on <pre> only, and render pre > code as a clean block so
   inline <code> padding/background never paints over the previous line. */
.msg-card[data-mtype="prompt"] .msg-content pre,
.msg-card.card-in[data-mtype="note"] .msg-content pre,
.msg-card.card-in[data-mtype="prompt"] .msg-content pre {
  background: #16161e !important;
  border-radius: 0 !important;
  padding: 8px 10px !important;
  border: 0 !important;
}
.msg-content pre > code {
  display: block !important;
  padding: 0 !important;
  margin: 0 !important;
  background: transparent !important;
  line-height: var(--craft-code-lh) !important;
}

/* ── 4. Card edges: no frame or selection stripe ──────────────────── */
.msg-card {
  border-top-color: transparent !important;
  border-right-color: transparent !important;
  border-bottom-color: transparent !important;
  border-left-color: transparent !important;
  box-shadow: none !important;
}
.editable[data-sm="unselected"] .msg-card { border-left-color: transparent !important; }

/* Widths stay 1px throughout so nothing shifts when the stripe appears */
.msg-card.card-out {
  border-left-color: transparent !important;
  border-top-left-radius: 0 !important;
  border-top-right-radius: 0 !important;
}
.msg-card.card-in:has(+ .msg-card.card-out:not(:empty)) {
  border-bottom-left-radius: 0 !important;   /* square only once an output exists */
  border-bottom-right-radius: 0 !important;
}

/* ── 5. Code outputs: header out of flow, actions parked top-right ──── */
.card-out[data-mtype="code"] { position: relative !important; }
.card-out[data-mtype="code"] > .uk-card-header {
  position: absolute !important;
  top: 0; right: 4px;
  z-index: 5;
  width: auto !important;
  padding: 0 !important;
  background: transparent !important;
  border: 0 !important;
}
.card-out[data-mtype="code"] > .uk-card-header nav { width: auto !important; }
.card-out[data-mtype="code"] > .uk-card-header nav > div.grow { display: none !important; }

/* ── 6. Composer: square, flush to the bottom, card-coloured ────────── */
.uk-container > div.flex-none.uk-card.uk-card-body {
  border-radius: 0 !important;
  padding-bottom: 0 !important;
  background-color: #1f2030 !important;
}
.monaco-editor,                          /* Monaco lives ONLY in the composer */
.monaco-editor-background,
.monaco-editor .margin {
  background-color: #1f2030 !important;
}

/* ── 7. Top nav: auto-hiding overlay, floating controls ─────────────── */
.uk-container > nav.monster-navbar {
  position: absolute !important;         /* out of flow: scroller keeps the 44px */
  top: 0; left: 0; right: 0;
  z-index: 40;
  height: 31px !important;               /* 0.7 x original */
  min-height: 0 !important;
  flex-wrap: nowrap !important;
  padding-block: 0 !important;
  background: transparent !important;
  border: 0 !important;
  opacity: 0;
  pointer-events: none;                  /* hidden strip never eats clicks */
  transition: opacity 0.18s ease;
}
/* Visibility is driven by CRAFT.js, which decides when to pin the bar open:
   pointer inside the trigger band, over the nav, or inside an open uk-drop.
   :focus-within stays so keyboard tabbing still reveals it. */
.uk-container > nav.monster-navbar.nav-shown,
.uk-container > nav.monster-navbar:focus-within {
  opacity: 1;
  pointer-events: auto;
}

/* Children inert only while the bar is hidden */
.uk-container > nav.monster-navbar > * { pointer-events: none; }
.uk-container > nav.monster-navbar.nav-shown > *,
.uk-container > nav.monster-navbar:focus-within > * { pointer-events: auto; }

.uk-container > nav.monster-navbar h2 {
  font-size: 1.05rem !important;
  line-height: 1.2 !important;
}
.uk-container > nav.monster-navbar span.rounded-full.size-4 {
  display: none !important;              /* the round status dot */
}
/* The brand link (📚 solveit) is removed outright; the dialog name moves into
   the file tree header — CRAFT.js relocates the live #dlg-name element there,
   so this navbar-scoped rule hides only the spot it leaves behind (and keeps
   the navbar clean until the tree is first opened). */
.uk-container > nav.monster-navbar a:has(> h1),
.uk-container > nav.monster-navbar h2#dlg-name { display: none !important; }

/* No containers: strip the action wrapper, the chips, and the <select> frame */
.uk-container > nav.monster-navbar > div.ml-auto,
.uk-container > nav.monster-navbar .uk-btn,
.uk-container > nav.monster-navbar .uk-btn-group,
.uk-container > nav.monster-navbar .uk-btn-group > *,
.uk-container > nav.monster-navbar select,
.uk-container > nav.monster-navbar input:checked + label.uk-btn,
.uk-container > nav.monster-navbar .uk-btn.uk-active {
  background: transparent !important;
  background-color: transparent !important;
  border: 0 !important;
  border-color: transparent !important;
  border-radius: 0 !important;
  box-shadow: none !important;
  outline: 0 !important;                 /* UIkit draws its chips as inset/groove outlines */
  outline-color: transparent !important;
  padding: 0 !important;
}
.uk-container > nav.monster-navbar .uk-btn,
.uk-container > nav.monster-navbar select { color: #cfc9c2 !important; }
.uk-container > nav.monster-navbar select { padding-inline: 2px !important; }
.uk-container > nav.monster-navbar .uk-btn:hover { color: #ffffff !important; }

/* Active state lives in the glyph colour plus a light glow, not a chip.
   The app nests each toggle's hidden checkbox inside its own label (default
   code, tools, diff) or button (brain), so :has() is what actually matches. */
.uk-container > nav.monster-navbar :is(label, button):has(input:checked) {
  color: #7aa2f7 !important;
  filter: drop-shadow(0 0 4px #7aa2f7cc) drop-shadow(0 0 10px #7aa2f766);
}
.uk-container > nav.monster-navbar .uk-btn.uk-active { color: #7aa2f7 !important; }

/* Dropdown list is OS-rendered once the select is transparent */
.uk-container > nav.monster-navbar select option {
  background: #16161e !important;
  color: #cfc9c2 !important;
}

/* Glyphs stay legible over whatever scrolls beneath */
.uk-container > nav.monster-navbar svg,
.uk-container > nav.monster-navbar select,
.uk-container > nav.monster-navbar h2 {
  filter: drop-shadow(0 1px 2px #16161e) drop-shadow(0 0 4px #16161e);
}
/* Input code uses --craft-code (section 2); line-height comes from the global rule there */
.card-in .msg-content pre {
  font-size: var(--craft-code);
}
.editable.dot-prompt .msg-card.card-in .msg-body {
  display: none !important;
}

/* ── Compact disclosures: borderless, tight stacking ────────────── */
/* Expandable boxes (tool calls, [data-panel] details) lose their frame
   and blend into the containing cell. Collapsed boxes stack with almost
   no vertical space; a hint of inset remains only while open. */
.msg-content details[data-panel],
.msg-content .tool-usage-details {
  border: 0 !important;
  background: transparent !important;
  box-shadow: none !important;
}
.msg-content details[data-panel]:not([open]) {
  margin: 0 !important;
  padding: 0.05em 0 !important;
}
.msg-content details[data-panel] > summary,
.msg-content .tool-usage-details > summary {
  margin: 0 !important;
  padding: 0.05em 0 !important;
}
.msg-content details[data-panel][open] {
  margin: 0.15em 0 !important;
  padding: 0.1em 0 0.1em 0.6em !important;
}
.msg-content .tool-usage-details[open] { padding: 0.15em 0 !important; }

/* ── Compact typography: minimal line and block spacing ─────────── */
/* Blank lines in source become paragraph breaks in rendered markdown:
   their gap stays slightly larger than the gap between wrapped lines. */
.msg-content .prose { line-height: var(--craft-prose-lh) !important; }
.msg-content .prose > * + * { margin-top: 0.15em !important; }
.msg-content .prose h1, .msg-content .prose h2, .msg-content .prose h3,
.msg-content .prose h4, .msg-content .prose h5, .msg-content .prose h6 {
  margin: 0.3em 0 0.12em !important;
  line-height: 1.08 !important;
}
.msg-content .prose h1 { font-size: 1.65em !important; }
.msg-content .prose h2 { font-size: 1.32em !important; }
.msg-content .prose h3 { font-size: 1.15em !important; }
.msg-content .prose h4 { font-size: 1.05em !important; }
.msg-content .prose h5, .msg-content .prose h6 { font-size: 1em !important; }
.msg-content .prose ul, .msg-content .prose ol { margin: 0.125em 0 !important; padding-left: 1.35em !important; }
.msg-content .prose li { margin: 0.05em 0 !important; }
.msg-content .prose li > p { margin: 0.05em 0 !important; }
.msg-content .prose hr { margin: 0.5em 0 !important; }
.msg-content .prose blockquote, .msg-content .prose pre,
.msg-content .prose table, .msg-content .prose figure { margin: 0.3em 0 !important; }

/* ── Launcher (⌘⇧P) ─────────────────────────────────────────────────── */
#solveit-launcher { position: fixed; inset: 0; z-index: 1200; display: none; justify-content: center; align-items: flex-start; padding-top: 12vh; background: #0008; }
#solveit-launcher.open { display: flex; }
#solveit-launcher .solveit-launcher-card { width: min(640px, 92vw); background: #1f2030; border: 1px solid #ffffff1a; border-radius: 8px; box-shadow: 0 16px 48px #000c; overflow: hidden; }
#solveit-launcher input { box-sizing: border-box; width: 100%; padding: 10px 14px; border: 0; border-bottom: 1px solid #ffffff14; outline: 0; background: transparent; color: #cfc9c2; font-size: 14px; }
#solveit-launcher ul { max-height: 50vh; overflow-y: auto; margin: 0; padding: 4px; list-style: none; }
#solveit-launcher li { display: flex; align-items: baseline; gap: 10px; padding: 5px 10px; border-radius: 4px; color: #cfc9c2; font-size: 12.5px; cursor: pointer; }
#solveit-launcher li[aria-selected="true"] { background: #2b2e43; }
#solveit-launcher li .label { flex: 0 1 auto; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
#solveit-launcher li .detail { flex: 1 1 0%; min-width: 0; color: #cfc9c266; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
#solveit-launcher li .hint { flex: 0 0 auto; color: #cfc9c288; font: 11px monospace; }
#solveit-launcher li.kind-dialog .hint { color: #bb9af7; }
#solveit-launcher li.empty { color: #cfc9c266; cursor: default; }

/* Encode Sans: all non-code UI and prose; remote Google Fonts variable TTF, with a system sans-serif fallback. */
@font-face { font-family: 'Craft Encode Sans'; font-style: normal; font-weight: 100 900; font-stretch: 75% 125%; font-display: swap; src: url('https://cdn.jsdelivr.net/gh/google/fonts@main/ofl/encodesans/EncodeSans%5Bwdth%2Cwght%5D.ttf') format('truetype'); }
:root { --craft-prose-family: 'Craft Encode Sans', -apple-system, sans-serif; }
html, body { font-family: var(--craft-prose-family) !important; font-weight: 360; font-stretch: 75%; font-feature-settings: "onum" 1; }
button, input, select, textarea { font-family: inherit; font-stretch: inherit; font-feature-settings: inherit; }
.msg-content, .msg-content .prose { font-family: var(--craft-prose-family) !important; font-weight: 360 !important; font-stretch: 75% !important; font-feature-settings: "onum" 1 !important; }
.msg-content .prose :is(strong, b, th, h1, h2, h3, h4, h5, h6) { font-weight: 650 !important; }

/* Iosevka: code only; remote font files, with a system monospace fallback. */
@font-face { font-family: 'Craft Iosevka'; font-style: normal; font-weight: 400; font-display: swap; src: url('https://iosevka-webfonts.github.io/iosevka/WOFF2/Iosevka-Regular.woff2') format('woff2'); }
@font-face { font-family: 'Craft Iosevka'; font-style: italic; font-weight: 400; font-display: swap; src: url('https://iosevka-webfonts.github.io/iosevka/WOFF2/Iosevka-Italic.woff2') format('woff2'); }
@font-face { font-family: 'Craft Iosevka'; font-style: normal; font-weight: 700; font-display: swap; src: url('https://iosevka-webfonts.github.io/iosevka/WOFF2/Iosevka-Bold.woff2') format('woff2'); }
@font-face { font-family: 'Craft Iosevka'; font-style: italic; font-weight: 700; font-display: swap; src: url('https://iosevka-webfonts.github.io/iosevka/WOFF2/Iosevka-BoldItalic.woff2') format('woff2'); }
:root { --craft-code-family: 'Craft Iosevka', monospace; }
.msg-content pre, .msg-content code { font-family: var(--craft-code-family) !important; font-weight: 400 !important; font-stretch: normal !important; font-feature-settings: normal !important; }
pre, code, kbd, samp, tt, .font-mono, .monaco-editor, .solveit-tree-shortcut, .solveit-file-tree-path, .solveit-tree-dots, #solveit-launcher li .hint { font-stretch: normal !important; font-feature-settings: normal !important; }


/* Compact dialog headers only: preserve action-icon sizes and editor layout. */
.msg-card > .uk-card-header,
.msg-card > .uk-card-header > nav.monster-navbar {
  padding-block: 0 !important;
  min-height: 0 !important;
}
html[data-craft-inserting] #full_editor, #full_editor.craft-composer-idle { display: none !important; }
.msg-card[data-mtype="note"] { background: transparent !important; }



.editable.craft-chain-next { margin-bottom: 0 !important; padding-bottom: 0 !important; border-bottom-left-radius: 0 !important; border-bottom-right-radius: 0 !important; }
.editable.craft-chain-prev { margin-top: 0 !important; padding-top: 0 !important; border-top-left-radius: 0 !important; border-top-right-radius: 0 !important; }
.editable.craft-chain-next > .msg-card { border-bottom-left-radius: 0 !important; border-bottom-right-radius: 0 !important; }
.editable.craft-chain-prev > .msg-card { border-top-left-radius: 0 !important; border-top-right-radius: 0 !important; }
.editable.craft-chain-prev > .card-in:is([data-mtype="prompt"], [data-mtype="code"]) > .uk-card-header [data-tokens] { display: none !important; }

.editable:not(.craft-chain-next) { margin-bottom: 25px !important; }
.editable > .msg-card[data-mtype="prompt"] { background: #1f2030 !important; }
.editable > .card-out[data-mtype="prompt"] { padding-top: 0px !important; }
.editable.craft-chain-prev > .card-in[data-mtype="prompt"] { padding-top: 0px !important; }
.editable > .card-out[data-mtype="code"] { background: transparent !important; padding-top: 2px !important; padding-bottom: 2px !important; }
.editable.craft-chain-next:has(> .card-in[data-mtype="code"]) { margin-bottom: 12px !important; }
.editable.craft-chain-prev > .card-in[data-mtype="code"] { border-top: 0 !important; }

.msg-card[data-mtype="code"] > .uk-card-header nav > div.grow > p { opacity: 0; }
.editable:is(:hover, :focus-within) > .msg-card[data-mtype="code"] > .uk-card-header nav > div.grow > p { opacity: 1; }
@media (hover: none) {
  .msg-card[data-mtype="code"] > .uk-card-header nav > div.grow > p { opacity: 1; }
}

.msg-card[data-mtype="prompt"] > .uk-card-header nav > div.grow > p::before { content: "You"; color: #a9b1d6 !important; font-size: 11px; }
.msg-card.card-out[data-mtype="prompt"] > .uk-card-header nav > div.grow > p::before { content: "Assistant"; }
.editable > .msg-card[data-mtype="prompt"] > .uk-card-header [data-tokens] { display: inline !important; font-size: 0 !important; }
.msg-card[data-mtype="prompt"] > .uk-card-header [data-tokens]::after { content: " · " attr(data-tokens) " tokens"; font-size: 11px; }
.msg-card[data-mtype="prompt"] > .uk-card-header nav > div.grow > p > span { opacity: 0; }
.msg-card[data-mtype="prompt"]:is(:hover, :focus-within) > .uk-card-header nav > div.grow > p > span { opacity: 1; }
.msg-card.card-in[data-mtype="prompt"] .msg-content,
.msg-card.card-in[data-mtype="prompt"] .msg-content .prose { color: #e0dcd5 !important; }
@media (hover: none) {
  .msg-card[data-mtype="prompt"] > .uk-card-header nav > div.grow > p > span { opacity: 1; }
}

#dialog-container { padding-top: 24px !important; }
#dialog-container .editable { border-radius: 0 !important; }
#dialog-container > .editable[data-sm]:not([data-sm="unselected"]) { position: relative; z-index: 2; box-shadow: none !important; }
#dialog-container > .editable[data-sm]:not([data-sm="unselected"])::after { content: ""; position: absolute; inset: 0; border: 2px solid oklch(62.3% 0.214 259.815); border-radius: 0 !important; pointer-events: none; z-index: 100; }

.editable.dot-prompt-response:not([data-editing="true"]) > .card-in { display: none !important; }
.editable.dot-prompt-response:not([data-editing="true"]) > .card-out { padding-top: 0 !important; }
.editable.dot-prompt-response.craft-chain-prev:not([data-editing="true"]) > .card-out { padding-top: 16px !important; }
.editable.dot-prompt-response:not(.craft-chain-prev):not([data-editing="true"]) > .card-out { border-top-left-radius: 8px !important; border-top-right-radius: 8px !important; }

.craft-action-source, .craft-old-action-toggle { display: none !important; }
.msg-card .craft-action-trigger { opacity: 0; pointer-events: none; width: 28px; height: 22px; flex: 0 0 28px; padding: 0 !important; background: transparent !important; border: 0 !important; color: #a9b1d6; font-size: 20px; line-height: 20px; cursor: pointer; }
.msg-card:hover .craft-action-trigger, .msg-card:focus-within .craft-action-trigger, .craft-action-trigger[aria-expanded="true"] { opacity: 1; pointer-events: auto; }
.craft-action-menu { position: fixed; inset: auto; margin: 0; width: 260px; max-width: calc(100vw - 16px); max-height: min(70vh, 540px); overflow: auto; padding: 6px; background: #242638; color: #cfc9c2; border: 1px solid #ffffff24; border-radius: 6px; box-shadow: 0 8px 28px #0008; }
.craft-action-menu button { display: block; width: 100%; padding: 7px 9px; text-align: left; background: transparent; color: inherit; border: 0; border-radius: 3px; font: inherit; cursor: pointer; }
.craft-action-menu button:is(:hover,:focus-visible) { background: #34374f; outline: 1px solid #7aa2f7; }
.craft-action-menu button:disabled { opacity: .4; cursor: default; }
.craft-action-menu button[data-danger] { color: #f7768e; }
@media (hover: none) { .msg-card .craft-action-trigger { opacity: 1; pointer-events: auto; } }

/* Code-cell header floats as a hover chip over the top-right corner, beside the copy button, so hovering never changes layout. */
@media (hover: hover) {
  .editable > .card-in[data-mtype="code"] > .uk-card-header {
    position: absolute !important;
    top: 0;
    right: 30px;
    z-index: 5;
    width: auto !important;
    max-width: calc(100% - 40px);
    padding: 0 2px 0 4px !important;
    background: #1f2030 !important;
    border: 0 !important;
    border-radius: 0 0 4px 4px;
    opacity: 0;
    pointer-events: none;
    transition: opacity .12s ease;
  }
  .editable:is(:hover, [data-editing="true"]) > .card-in[data-mtype="code"] > .uk-card-header,
  .editable > .card-in[data-mtype="code"] > .uk-card-header:is(:focus-within, :has(.craft-action-trigger[aria-expanded="true"])) { opacity: 1; pointer-events: auto; }
  .editable > .card-in[data-mtype="code"] > .uk-card-header nav { width: auto !important; flex-wrap: nowrap !important; }
  .editable > .card-in[data-mtype="code"] > .uk-card-header nav > div.grow { flex: none; }
}
#dialog-container .editable > .msg-card:not(.collapsed),
#dialog-container .editable > .msg-card:not(.collapsed) > .msg-body {
  max-height: none !important;
  overflow: visible !important;
}
#dialog-container .editable > .msg-card:not(.collapsed) .msg-content :is(pre, table) {
  max-height: none !important;
  overflow: visible !important;
  overflow-wrap: anywhere;
}
#dialog-container .editable > .msg-card:not(.collapsed) .msg-content pre { white-space: pre-wrap; }
.editable > .card-in[data-mtype="code"]:not(.collapsed) .copy-wrap > button { top: 0; max-height: 100%; }


.editable.craft-omitted { display: none !important; }
.craft-omit-marker { display: block; flex: none; width: fit-content; margin: 2px 0; padding: 1px 6px; border: 0; border-radius: 3px; background: transparent; color: #9a9eb5; font: inherit; font-size: 11px; line-height: 18px; cursor: pointer; }
.craft-omit-marker:hover { color: #cfc9c2; }
.craft-omit-marker:focus-visible { outline: 1px solid #7aa2f7; outline-offset: 1px; }


.msg-card.craft-part-hidden { display: none !important; }
.craft-part-restore { justify-self: start; margin: 0; padding: 0 6px; border: 0; border-radius: 3px; background: transparent; color: #9a9eb5; font: inherit; line-height: 18px; cursor: pointer; }
.craft-part-restore:hover { color: #cfc9c2; }
.craft-part-restore:focus-visible { outline: 1px solid #7aa2f7; }

/* ── 13. Tokyo Night cohesion for Solveit-native overlays & panels ───── */
body:has(.uk-modal.uk-open, .hk-overlay.open, #solveit-launcher.open) :is(#craft-insert-preview, #craft-mobile-toggle) {
  pointer-events: none;
  opacity: 0.2;
}

/* Find / Replace floating bar */
#find-box {
  background: #16161e !important;
  border: 1px solid #292e42 !important;
  border-radius: 8px !important;
  box-shadow: 0 12px 32px rgba(0, 0, 0, 0.55) !important;
  color: #cfc9c2 !important;
  font-feature-settings: normal !important;
}
#find-box .uk-btn,
#find-box button,
#find-box label.uk-btn {
  background: #1f2030 !important;
  border-color: #292e42 !important;
  color: #a9b1d6 !important;
}
#find-box .uk-btn:hover,
#find-box button:hover,
#find-box label.uk-btn:hover {
  background: #292e42 !important;
  color: #e0e0e8 !important;
}
#find-box label.uk-btn:has(input:checked) {
  background: rgba(122, 162, 247, 0.22) !important;
  border-color: rgba(122, 162, 247, 0.65) !important;
  color: #7aa2f7 !important;
}
#find-box input[type="text"],
#find-box input[type="search"],
#find-box input.uk-input {
  background: #1f2030 !important;
  border: 1px solid #292e42 !important;
  color: #cfc9c2 !important;
  border-radius: 5px !important;
}
#find-box input:focus {
  border-color: #7aa2f7 !important;
  outline: none !important;
  box-shadow: 0 0 0 1px rgba(122, 162, 247, 0.35) !important;
}

/* Settings & Solveit UIkit modals */
.uk-modal .uk-modal-dialog {
  background: #16161e !important;
  border: 1px solid #292e42 !important;
  border-radius: 10px !important;
  box-shadow: 0 20px 56px rgba(0, 0, 0, 0.65) !important;
  color: #cfc9c2 !important;
  font-feature-settings: normal !important;
}
.uk-modal .uk-modal-dialog :is(h1, h2, h3, h4, button, a, label, th, td, p) {
  font-feature-settings: normal !important;
}
.uk-modal .uk-modal-title {
  color: #e0e0e8 !important;
}
.uk-modal .uk-tab-alt {
  background: #1f2030 !important;
  border: 1px solid #292e42 !important;
  border-radius: 6px !important;
}
.uk-modal .uk-tab-alt > li > a {
  color: #9a9eb5 !important;
}
.uk-modal .uk-tab-alt > li.uk-active > a {
  background: #292e42 !important;
  color: #e0e0e8 !important;
  border-radius: 4px !important;
}
.uk-modal .uk-modal-dialog .uk-btn-primary,
.uk-modal .uk-modal-dialog a.uk-btn-primary {
  background: rgba(122, 162, 247, 0.16) !important;
  border: 1px solid rgba(122, 162, 247, 0.45) !important;
  color: #c0caf5 !important;
}
.uk-modal .uk-modal-dialog .uk-btn-primary:hover,
.uk-modal .uk-modal-dialog a.uk-btn-primary:hover {
  background: rgba(122, 162, 247, 0.28) !important;
  border-color: #7aa2f7 !important;
  color: #ffffff !important;
}
.uk-modal .uk-modal-dialog .uk-btn-default,
.uk-modal .uk-modal-dialog .uk-modal-close {
  background: #1f2030 !important;
  border: 1px solid #292e42 !important;
  color: #a9b1d6 !important;
}
.uk-modal .uk-modal-dialog .uk-btn-default:hover,
.uk-modal .uk-modal-dialog .uk-modal-close:hover {
  background: #292e42 !important;
  color: #e0e0e8 !important;
}
.uk-modal .uk-modal-dialog hr,
.uk-modal .uk-modal-dialog .border-b,
.uk-modal .uk-modal-dialog .border-t {
  border-color: #292e42 !important;
}

/* Variable / TOC sidebar (#vars-sb) */
aside#vars-sb {
  background: #16161e !important;
  border-left: 1px solid #292e42 !important;
  padding-left: 12px !important;
  color: #cfc9c2 !important;
  font-feature-settings: normal !important;
}
aside#vars-sb * {
  font-feature-settings: normal !important;
}
aside#vars-sb .uk-card,
aside#vars-sb .border,
aside#vars-sb .border-b,
aside#vars-sb hr {
  background: transparent !important;
  border-color: #292e42 !important;
}
aside#vars-sb a:hover,
aside#vars-sb button:hover {
  color: #7aa2f7 !important;
}
aside#vars-sb .uk-close,
aside#vars-sb button[aria-label="Close"] {
  background: #1f2030 !important;
  border: 1px solid #292e42 !important;
  color: #a9b1d6 !important;
}

/* Minimalist empty-dialog cue (only when no cells exist) */
#dialog-container:not(:has(> .editable)) {
  position: relative;
}
#dialog-container:not(:has(> .editable))::before {
  content: "Empty dialog — type below or press Shift+I to compose";
  position: absolute;
  inset: 0;
  display: grid;
  place-items: center;
  color: rgba(207, 201, 194, 0.45);
  font-size: 12px;
  font-feature-settings: normal;
  pointer-events: none;
}


/* ── 14. Minimalist cell state indicators (Pinned, Exported, Hidden) ── */
/* Pinned: quiet Tokyo Night gold 2px left accent */
.editable:has(button[hx-post="/toggle_pin_"] use[href="#lc-pin"]) > .card-in {
  box-shadow: inset 2px 0 0 #e0af68 !important;
}
/* Exported: quiet Tokyo Night cyan 2px left accent */
.editable:has(button[hx-post="/toggle_export_"] use[href="#lc-bookmark-check"]) > .card-in {
  box-shadow: inset 2px 0 0 #7dcfff !important;
}
/* Both Pinned and Exported: dual 2px+2px left accent */
.editable:has(button[hx-post="/toggle_pin_"] use[href="#lc-pin"]):has(button[hx-post="/toggle_export_"] use[href="#lc-bookmark-check"]) > .card-in {
  box-shadow: inset 2px 0 0 #e0af68, inset 4px 0 0 #7dcfff !important;
}
/* Hidden from AI (skipped): dimmed at rest, recovers on hover/focus, dashed left bar */
.editable:has(button[hx-post="/toggle_skip_"] use[href="#lc-eye-off"]) {
  opacity: 0.52;
  transition: opacity 0.15s ease;
}
.editable:has(button[hx-post="/toggle_skip_"] use[href="#lc-eye-off"]):is(:hover, :focus-within, [data-sm="primary"], [data-sm="secondary"]) {
  opacity: 0.9;
}
.editable:has(button[hx-post="/toggle_skip_"] use[href="#lc-eye-off"]) > .card-in {
  border-left: 2px dashed rgba(154, 158, 181, 0.5) !important;
}


/* ── 15. Markdown task lists & Raw cell polish ──────────────────────── */
.msg-content ul.task-list {
  list-style: none !important;
  padding-left: 0.15em !important;
}
.msg-content ul.task-list > li {
  display: flex;
  align-items: baseline;
  gap: 0.5em;
  list-style: none !important;
}
.msg-content ul.task-list > li::marker {
  content: none !important;
}
.msg-content ul.task-list input[type="checkbox"] {
  appearance: none;
  -webkit-appearance: none;
  flex: none;
  width: 11px;
  height: 11px;
  margin: 0;
  transform: translateY(1.5px);
  border: 1px solid #565f89;
  border-radius: 2.5px;
  background: #16161e;
  opacity: 1 !important;
  position: relative;
}
.msg-content ul.task-list input[type="checkbox"]:checked {
  background: rgba(122, 162, 247, 0.22);
  border-color: #7aa2f7;
}
.msg-content ul.task-list input[type="checkbox"]:checked::after {
  content: "";
  position: absolute;
  left: 3px;
  top: 0.5px;
  width: 3.5px;
  height: 6.5px;
  border: solid #7aa2f7;
  border-width: 0 1.5px 1.5px 0;
  transform: rotate(45deg);
}

/* Raw cells: subtle recessed dashed frame, disable ligatures so YAML `---` stays literal */
.editable > .card-in[data-mtype="raw"] {
  background: #1a1b2a !important;
  border: 1px dashed #292e42 !important;
  border-radius: 4px !important;
}
.editable > .card-in[data-mtype="raw"] .msg-content :is(pre, code) {
  font-variant-ligatures: none;
}

/* ── 16. Cohesive Tabular Output (Markdown, pandas, polars, DuckDB) ─── */
.editable > .msg-card,
.editable > .msg-card > .msg-body,
.editable > .msg-card .msg-content,
.editable > .msg-card .msg-content .prose { min-width: 0; max-width: 100%; }
.craft-table-wrap {
  position: relative;
  width: fit-content;
  max-width: 100%;
  max-height: 480px;
  overflow: auto;
  -webkit-overflow-scrolling: touch;
  margin: 0.4em 0 !important;
  border: 1px solid #292e42;
  border-radius: 5px;
  background: #1a1b2a;
}
.craft-table-wrap:focus-visible {
  outline: 1.5px solid #7aa2f7;
  outline-offset: 1px;
}
.msg-content table {
  border-collapse: separate !important;
  border-spacing: 0 !important;
  width: auto !important;
  margin: 0 !important;
  border: 0 !important;
  background: transparent !important;
  font-family: var(--craft-prose-family) !important;
  font-size: 11px !important;
  line-height: 1.4 !important;
  font-feature-settings: "tnum" 1, "lnum" 1, "onum" 0, "c2sc" 0 !important;
  font-variant-numeric: tabular-nums lining-nums !important;
  color: #cfc9c2 !important;
}
.msg-content table :is(th, td) {
  font-feature-settings: "tnum" 1, "lnum" 1, "onum" 0, "c2sc" 0 !important;
  font-variant-numeric: tabular-nums lining-nums !important;
  border-top: 0 !important;
  border-left: 0 !important;
  border-right: 0 !important;
}
.msg-content table thead {
  position: sticky;
  top: 0;
  z-index: 2;
}
.msg-content table thead tr {
  text-align: left !important;
}
.msg-content table thead th {
  background: #16161e !important;
  color: #c0caf5 !important;
  font-weight: 600 !important;
  padding: 5px 10px !important;
  border-bottom: 1px solid #3b4261 !important;
  text-align: left !important;
  vertical-align: bottom !important;
  white-space: nowrap;
}
.msg-content table thead tr:not(:last-child) th {
  border-bottom-color: #292e42 !important;
}
.msg-content table tbody th {
  background: #181926 !important;
  color: #9aa5ce !important;
  font-weight: 500 !important;
  padding: 4px 10px !important;
  border-bottom: 1px solid #24283b !important;
  border-right: 1px solid #292e42 !important;
  text-align: left !important;
  vertical-align: top !important;
  white-space: nowrap;
}
.msg-content table tbody td {
  padding: 4px 10px !important;
  border-bottom: 1px solid #24283b !important;
  color: #cfc9c2 !important;
  text-align: left !important;
  vertical-align: top !important;
}
.msg-content table tbody tr:last-child > :is(th, td) {
  border-bottom: 0 !important;
}
.msg-content table tbody tr:hover > td {
  background: rgba(122, 162, 247, 0.06) !important;
}
.msg-content table :is(th, td).craft-num {
  text-align: right !important;
  white-space: nowrap;
}
.msg-content table thead th.craft-sort {
  cursor: pointer;
  user-select: none;
  padding-right: 18px !important;
  position: relative;
}
.msg-content table thead th.craft-sort:hover {
  background: #1d1f2e !important;
  color: #e1e4e8 !important;
}
.msg-content table thead th.craft-sort:focus-visible {
  outline: 1.5px solid #7aa2f7;
  outline-offset: -1.5px;
}
.msg-content table thead th.craft-sort::after {
  content: "";
  position: absolute;
  right: 5px;
  top: 50%;
  transform: translateY(-50%);
  font-size: 8px;
  color: #7aa2f7;
  opacity: 0;
}
.msg-content table thead th.craft-sort[aria-sort="ascending"]::after {
  content: "\25B2";
  opacity: 1;
}
.msg-content table thead th.craft-sort[aria-sort="descending"]::after {
  content: "\25BC";
  opacity: 1;
}
.craft-table-copy {
  position: sticky;
  float: right;
  top: 3px;
  right: 4px;
  z-index: 4;
  margin-left: -60px;
  padding: 1px 6px;
  border: 1px solid #292e42;
  border-radius: 3px;
  background: #16161ee6;
  color: #9aa5ce;
  font-family: var(--craft-prose-family);
  font-size: 9.5px;
  line-height: 16px;
  cursor: pointer;
  opacity: 0;
  transition: opacity 0.12s ease, color 0.12s ease, border-color 0.12s ease;
}
.craft-table-wrap:hover .craft-table-copy,
.craft-table-wrap:focus-within .craft-table-copy,
.craft-table-copy.is-copied {
  opacity: 1;
}
.craft-table-copy:hover {
  color: #e1e4e8;
  border-color: #7aa2f788;
}
.craft-table-copy.is-copied {
  color: #9ece6a;
  border-color: #9ece6a66;
}
/* Light theme table overrides */
html:not(.dark) .craft-table-wrap { border-color: #d0d7de; background: #ffffff; }
html:not(.dark) .msg-content table { color: #24292e !important; }
html:not(.dark) .msg-content table thead th { background: #f6f8fa !important; color: #24292e !important; border-bottom-color: #d0d7de !important; }
html:not(.dark) .msg-content table tbody th { background: #f6f8fa !important; color: #57606a !important; border-bottom-color: #eaeef2 !important; border-right-color: #d0d7de !important; }
html:not(.dark) .msg-content table tbody td { color: #24292e !important; border-bottom-color: #eaeef2 !important; }
html:not(.dark) .msg-content table tbody tr:hover > td { background: rgba(9, 105, 218, 0.05) !important; }
html:not(.dark) .msg-content table thead th.craft-sort:hover { background: #eaeef2 !important; }
html:not(.dark) .craft-table-copy { background: #f6f8fae6; border-color: #d0d7de; color: #57606a; }

/* ── 17. Accessibility: High-Contrast ANSI Traceback Remapping (dark) ─ */
html.dark .card-out pre span[style*="rgb(187,0,0)"],
html.dark .card-out pre span[style*="rgb(187, 0, 0)"],
html.dark .card-out pre span[style*="rgb(175,0,0)"],
html.dark .card-out pre span[style*="rgb(135,0,0)"] { color: #f7768e !important; }
html.dark .card-out pre span[style*="rgb(0,0,187)"],
html.dark .card-out pre span[style*="rgb(0, 0, 187)"],
html.dark .card-out pre span[style*="rgb(0,0,135)"],
html.dark .card-out pre span[style*="rgb(0, 0, 135)"] { color: #7aa2f7 !important; }
html.dark .card-out pre span[style*="rgb(0,135,0)"],
html.dark .card-out pre span[style*="rgb(0, 135, 0)"],
html.dark .card-out pre span[style*="rgb(0,187,0)"],
html.dark .card-out pre span[style*="rgb(0, 187, 0)"] { color: #9ece6a !important; }
html.dark .card-out pre span[style*="rgb(0,187,187)"],
html.dark .card-out pre span[style*="rgb(0, 187, 187)"],
html.dark .card-out pre span[style*="rgb(0,135,135)"],
html.dark .card-out pre span[style*="rgb(0, 135, 135)"] { color: #7dcfff !important; }
html.dark .card-out pre span[style*="rgb(187,187,0)"],
html.dark .card-out pre span[style*="rgb(187, 187, 0)"],
html.dark .card-out pre span[style*="rgb(135,95,0)"] { color: #e0af68 !important; }
html.dark .card-out pre span[style*="rgb(187,0,187)"],
html.dark .card-out pre span[style*="rgb(187, 0, 187)"] { color: #bb9af7 !important; }
html.dark .card-out pre span[style*="rgb(102,102,102)"],
html.dark .card-out pre span[style*="rgb(102, 102, 102)"] { color: #9aa5ce !important; }

/* ── Desktop border refinement, hover outlines & hover-edge drag-to-insert ── */
@media (hover: hover) {
  html:not(.craft-mobile) #dialog-container > .editable {
    border-radius: 6px !important;
    outline: 1px solid transparent;
    transition: outline-color .12s ease;
  }
  html:not(.craft-mobile) #dialog-container > .editable[data-sm="unselected"]:hover {
    outline-color: #ffffff14;
  }
}
html:not(.craft-mobile) #dialog-container > .editable[data-sm]:not([data-sm="unselected"])::after {
  border: 1.5px solid #7aa2f7c4;
  border-radius: 6px !important;
  box-shadow: 0 0 0 3px #7aa2f714;
}
html:not(.craft-mobile) #dialog-container > .editable:is([data-c2-editing="true"], [data-md-editing="true"], [data-editing="true"]) {
  position: relative; z-index: 3;
}
html:not(.craft-mobile) #dialog-container > .editable:is([data-c2-editing="true"], [data-md-editing="true"], [data-editing="true"])::after {
  content: ""; position: absolute; inset: 0;
  border: 1.5px solid #9ece6acc; border-radius: 6px !important;
  box-shadow: 0 0 0 3px #9ece6a18;
  pointer-events: none; z-index: 100;
}
html:not(.craft-mobile) #dialog-container > .editable.craft-desk-host {
  position: relative;
  overflow: visible !important; content-visibility: visible !important; contain: none !important;
}
html:not(.craft-mobile) #dialog-container > .editable.craft-desk-host:has(> .craft-desk-edge:is([data-near], :hover, :focus-visible, [data-active="true"]))::after {
  mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000);
  mask-position: 0 0, center bottom;
  mask-size: 100% 100%, 48px 6px;
  mask-repeat: no-repeat;
  mask-composite: exclude;
  -webkit-mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000);
  -webkit-mask-position: 0 0, center bottom;
  -webkit-mask-size: 100% 100%, 48px 6px;
  -webkit-mask-repeat: no-repeat;
  -webkit-mask-composite: xor;
}
.craft-desk-edge, #craft-desk-picker { display: none; }
html:not(.craft-mobile) .craft-desk-edge {
  display: flex; align-items: center; justify-content: center;
  position: absolute; left: 50%; z-index: 110;
  width: 64px; height: 22px; padding: 0; border: 0; background: transparent;
  transform: translateX(-50%);
  opacity: 0; pointer-events: none; transition: opacity .12s ease;
  user-select: none; cursor: grab;
}
html:not(.craft-mobile) .craft-desk-edge[data-edge="below"] { bottom: -11px; }
html:not(.craft-mobile) .craft-desk-edge:is([data-near], :hover, :focus-visible, [data-active="true"]) {
  opacity: 1; pointer-events: auto;
}
html:not(.craft-mobile) .craft-desk-edge-pill {
  display: inline-flex; align-items: center; gap: 3px;
  height: 18px; padding: 0 7px; border-radius: 999px;
  background: #1a1b26; border: 1px solid #7aa2f766; color: #7aa2f7;
  box-shadow: 0 2px 8px #0008;
  transition: transform .1s ease, background .1s ease, border-color .1s ease, color .1s ease;
}
html:not(.craft-mobile) .craft-desk-edge-kind { display: inline-flex; align-items: center; opacity: .75; }
html:not(.craft-mobile) .craft-desk-edge:hover .craft-desk-edge-pill,
html:not(.craft-mobile) .craft-desk-edge[data-active="true"] .craft-desk-edge-pill {
  background: #24283b; border-color: #7aa2f7; color: #c0caf5; transform: scale(1.06);
}
html:not(.craft-mobile) #craft-desk-picker {
  display: none; position: fixed; z-index: 1250;
  padding: 6px 8px 8px;
  background: #16161ef2; backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px);
  border: 1px solid #7aa2f755; border-radius: 10px;
  box-shadow: 0 12px 32px #000c; user-select: none;
}
html:not(.craft-mobile) #craft-desk-picker.open { display: block; }
html:not(.craft-mobile) #craft-desk-picker .craft-desk-picker-head {
  font: 500 10px/1 var(--craft-prose-family); color: #9aa5ce;
  text-align: center; margin-bottom: 6px; letter-spacing: .03em; text-transform: uppercase;
}
html:not(.craft-mobile) #craft-desk-picker .craft-desk-picker-opts { display: flex; gap: 5px; }
html:not(.craft-mobile) #craft-desk-picker button {
  display: inline-flex; align-items: center; gap: 5px;
  padding: 6px 9px; border: 1px solid #ffffff16; border-radius: 7px;
  background: #242638; color: #cfc9c2; font: 500 11.5px/1 var(--craft-prose-family);
  cursor: pointer; transition: transform .08s ease, background .08s ease, border-color .08s ease;
}
html:not(.craft-mobile) #craft-desk-picker button svg { color: #7aa2f7; }
html:not(.craft-mobile) #craft-desk-picker button kbd {
  font: 600 9.5px/1 var(--craft-code-family); color: #9aa5ce; opacity: .75;
  padding: 1px 4px; border-radius: 3px; background: #16161e;
}
html:not(.craft-mobile) #craft-desk-picker button:is(:hover, [data-hot]) {
  background: #7aa2f733; border-color: #7aa2f7; color: #ffffff; transform: translateY(-1px);
}

/* ── Mobile view toggle & Insert preview controls (moved from inline JS <style>) ── */
#craft-mobile-toggle { position: fixed; z-index: 955; width: 28px; height: 26px; display: grid; place-items: center; border: 0; border-radius: 4px; background: transparent; color: #9b9cab; opacity: .4; padding: 0; cursor: pointer; transition: opacity .15s ease, color .15s ease; }
#craft-mobile-toggle:hover, #craft-mobile-toggle:focus-visible, body:has(nav.monster-navbar.nav-shown) #craft-mobile-toggle { opacity: 1; }
#craft-mobile-toggle:hover, #craft-mobile-toggle:focus-visible { color: #e0e0e8; }
#craft-mobile-toggle:focus-visible { outline: 1px solid #7aa2f7; outline-offset: 2px; }
html.craft-mobile #craft-mobile-toggle { top: 8px !important; left: auto !important; right: 10px !important; z-index: 950; background: #16161ecc; border: 1px solid #ffffff1a; }

#dialog-container.craft-insert-preview { box-sizing: border-box; margin-right: 0 !important; min-width: 0; }
#craft-insert-preview { position: fixed; z-index: 955; width: 28px; height: 26px; display: grid; place-items: center; border: 0; border-radius: 4px; background: transparent; color: #9b9cab; opacity: .4; padding: 0; cursor: pointer; transition: opacity .15s ease, color .15s ease; }
#craft-insert-preview[hidden] { display: none; }
#craft-insert-preview:hover, #craft-insert-preview:focus-visible, body:has(nav.monster-navbar.nav-shown) #craft-insert-preview { opacity: 1; color: #e0e0e8; }
.craft-insert-placeholder { height: 3px; margin: 4px 0; border-radius: 999px; background: linear-gradient(90deg, #7aa2f722, #7aa2f7, #7dcfff, #7aa2f722); background-size: 200% 100%; animation: craft-ins-shimmer 1s linear infinite; pointer-events: none; }
@keyframes craft-ins-shimmer { from { background-position: 200% 0; } to { background-position: -200% 0; } }
#craft-insert-preview-menu { position: fixed; inset: auto; margin: 0; padding: 5px; border: 1px solid #414251; border-radius: 4px; background: #1f2030; color: #cfc9c2; font: 12px system-ui; }
#craft-insert-preview-menu button { display: flex; align-items: center; gap: 8px; width: 100%; padding: 5px 10px; border: 0; background: transparent; color: inherit; cursor: pointer; }
#craft-insert-preview-menu button:hover, #craft-insert-preview-menu button[aria-pressed=true] { background: #343649; }
#craft-insert-preview-menu small { display: block; padding: 4px 10px; color: #9b9cab; }


## Live `fastcdp` Verification

In [ ]:
#| live
import json
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    rungs = Rungs(page)

    async with rungs('command launcher open, filter, close'):
        await page.press('P', mod=True, shift=True)
        await page.wait_for_selector('#solveit-launcher.open input', timeout=3)
        await page.fill_text('#solveit-launcher.open input', 'line')
        await page.wait_for("document.querySelectorAll('#solveit-launcher.open ul[role=\"listbox\"] li').length >= 1", timeout=3)
        await page.press('Escape')
        await page.wait_for_selector('#solveit-launcher.open', present=False, timeout=3)

    async with rungs('action menu popover'):
        assert await page.sel_exists('#dialog-container > .editable .craft-action-trigger')
        await page.eval("document.querySelector('#dialog-container > .editable .craft-action-trigger').click()")
        await page.wait_for_selector(':popover-open', timeout=3)
        await page.press('Escape')
        await page.wait_for_selector(':popover-open', present=False, timeout=3)

    async with rungs('insert preview position and hotkey'):
        assert await page.eval(r"""(() => {
          const t = document.getElementById('craft-insert-preview'), sc = document.getElementById('dialog-container');
          return !!(t && sc && t.getBoundingClientRect().right <= sc.getBoundingClientRect().right);
        })()""")
        prev_type = await page.eval("window.craftInsertPreview.type()")
        await page.press('K', mod=True, shift=True)
        await page.wait_for("window.craftInsertPreview.type() === 'Note'", timeout=3)
        await page.eval(f"window.craftInsertPreview.choose({json.dumps(prev_type)})")

    async with rungs('desktop/mobile mode toggle'):
        assert await page.eval("window.craftMobile.active() === false")
        assert await page.sel_exists('#craft-mobile-toggle')
        await page.eval("window.craftMobile.toggle()")
        assert await page.eval("window.craftMobile.active() === true")
        await page.eval("window.craftMobile.setMode('auto')")
        assert await page.eval("window.craftMobile.active() === false")

    async with rungs('omit toggle and restore'):
        first_id = await page.eval(r"""(() => {
          const el = document.querySelector('#dialog-container > .editable[id]');
          if (!el) return null;
          window.selectMsg(window.$(el), {scroll: false});
          return el.id;
        })()""")
        assert first_id and await page.eval("window.craftOmit.toggle()") is True
        await page.wait_for_selector('.craft-omit-marker', timeout=3)
        assert await page.eval(f"document.getElementById({json.dumps(first_id)}).classList.contains('craft-omitted')")
        await page.eval("document.querySelector('.craft-omit-marker').click()")
        await page.wait_for_selector('.craft-omit-marker', present=False, timeout=3)
        assert await page.eval(f"!document.getElementById({json.dumps(first_id)}).classList.contains('craft-omitted')")

    assert not page_errors, f"Uncaught page errors: {page_errors}"